# **LAB511**: Create advanced Postgres-powered agentic apps with Azure HorizonDB

## Notebook 2 - Application Development

### Part 3.1: Introduction

**Welcome to the LAB511 Agent Application Development Notebook!** 🚀

Time to put on your AI engineer hat. In this notebook you'll wire up a legal research **agent** that can reason over the Washington State case law you just loaded into **Azure HorizonDB**, pull in live data from the web, and *remember* what it learns from every conversation.

**The stack you're about to light up:**

- 🧠 **Microsoft Agent Framework**: Microsoft's open-source SDK for building tool-using agents. You get an LLM, a clean `@tool` decorator for function calling, and conversation orchestration out of the box.
- 🤖 **Azure OpenAI**: GPT models for chat + reasoning, and an embedding model for turning text into vectors.
- 🐘 **Azure HorizonDB (Postgres)**: one database doing the work of four:
    - **Full-text search** with BM25 for fast keyword lookup
    - **Vector search** via `pgvector` for semantic similarity over case opinions
    - **Graph queries** via Apache AGE for relationships between cases, judges, and citations
    - **Operational data** in good old SQL tables
- 💾 **Mem0**: the long-term memory layer that lets your agent learn user preferences and recall facts across sessions. And the best part? **Mem0 stores its memory embeddings right back in HorizonDB using `pgvector`** - no extra vector database to run, no extra bill to pay. One Postgres to rule them all. 💍
- 🌦️ **External APIs**: because real legal arguments sometimes hinge on whether it was actually raining that day.

**Why this is cool:** instead of stitching together a vector DB, a graph DB, a memory service, and a relational DB, you're doing it all in **one Postgres instance**. Less ops, less latency, fewer things to break - and you still get to use the SQL you already know.

**Architecture Diagram**

> Here's what we're building today:

#### Part 3.1.1: Setup the Agent App Python imports

> **Note:** In your lab environment, we already have the PIP packages pre-deployed that are needed by the import statements in the following code block, so you do not need to perform any installs.

##### 🧠 *Technical Background Notes*

This cell pulls in everything our agent needs, grouped by job:

- **Async plumbing**: `nest_asyncio` patches Jupyter's event loop so we can `await` agent calls inline. `asyncio`, `time`, and `concurrent.futures` handle concurrency for parallel tool calls.
- **Standard library**: `os`, `re`, `sys`, `io`, `json`, and `datetime` for env vars, parsing, and general housekeeping.
- **Type hints**: `Annotated` (plus `Optional`, `List`, `Literal`) is the secret sauce: Agent Framework reads these annotations to auto-generate the JSON schema the LLM uses to call our tools.
- **Data + HTTP + DB**: `requests` for external APIs (hello, weather service), `pydantic.Field` for rich tool-parameter descriptions, `psycopg` as the modern PostgreSQL driver to talk to HorizonDB, and `dotenv` to load secrets from `.env`.
- **🧠 Microsoft Agent Framework** — `OpenAIChatClient` wires us up to Azure OpenAI, and the `@tool` decorator is how we turn ordinary Python functions into tools the agent can call.
- **💾 Mem0**: `Memory` is the long-term memory layer. It will store conversation-derived facts as embeddings in HorizonDB via `pgvector`: same database, new superpower.
- **🎨 UI**: `gradio` gives us a chat UI in a couple of lines, and `IPython.display.Markdown` keeps notebook output looking sharp.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. This will run the code and show the output below. Since these are just imports, there is nothing to show at the end other than a check mark showing success.

    > **Note:** The first time this code block is ran, it may take around 10-15 seconds.

In [ ]:
# Allow nested event loops (needed so async agent calls work inside Jupyter)
import nest_asyncio
nest_asyncio.apply()

# Standard library
import os
import re
import sys
import io
import json
import time
import asyncio
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# Type hints used by tool signatures (Annotated drives the JSON schema the agent sees)
from typing import Annotated, Optional, List, Literal

# Third-party: HTTP, data validation, database driver, env loading
import requests
from pydantic import Field
import psycopg
from dotenv import load_dotenv

# Agent Framework: LLM client + the @tool decorator that registers functions as callable tools
from agent_framework.openai import OpenAIChatClient
from agent_framework import tool

# Mem0: long-term memory layer for the agent (pgvector-backed in HorizonDB)
from mem0 import Memory

# UI + notebook display
import gradio as gr
from IPython.display import Markdown

#### Part 3.1.2: Load environment variables and configure connections

> **Note:** Your lab `.env` file is already populated with the right endpoints, keys, and database credentials. You do not need to edit anything.

##### 🧠 *Technical Background Notes*

This cell reads configuration from your `.env` file and stages it for the rest of the notebook. Keeping secrets out of source code is rule #1 of building cloud apps, and `load_dotenv(override=True)` makes sure your local values win over any stale shell variables.

- **Azure OpenAI settings**: `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_DEPLOYMENT`, and `AZURE_OPENAI_KEY` point the chat client at your GPT deployment. `AZURE_EMBED_DEPLOYMENT` is the embedding model used for vector search and Mem0 memories. `AZURE_API_VERSION` pins a known-good API contract.
- **HorizonDB connection (`DB_CONFIG`)**: a standard `psycopg` connection dict (host, dbname, user, password, port, sslmode). Note that we are deliberately pointing at the **read-only endpoint**: every tool the agent calls is a read query, so we offload that traffic to a read replica. This is a great pattern for keeping your primary node free for writes and analytics.
- **`sslmode=require`**: Azure HorizonDB enforces TLS, so this is non-negotiable and ensures your traffic is encrypted in transit.
- **Quick confirmation print**: we echo the deployment names and API version so you can sanity-check that the right environment was loaded before any agent code runs.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. Confirm the printed deployment names and API version match what is in your `.env` file.

    > **Note:** If you see a `KeyError`, your `.env` file is missing a value. Check it and re-run.

In [ ]:
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT   = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
AZURE_OPENAI_KEY        = os.environ["AZURE_OPENAI_KEY"]
AZURE_EMBED_DEPLOYMENT  = os.environ["AZURE_EMBED_DEPLOYMENT"]
AZURE_API_VERSION       = os.environ["AZURE_API_VERSION"]

DB_CONFIG = {
    # Note we are using the read-only endpoint for the agent as all queries
    # are read-only and is an example of offloading read workloads to a read replica
    "host":     os.environ["AZURE_PG_HOST"],
    "dbname":   os.environ["AZURE_PG_NAME"],
    "user":     os.environ["AZURE_PG_USER"],
    "password": os.environ["AZURE_PG_PASSWORD"],
    "port":     os.environ["AZURE_PG_PORT"],
    "sslmode":  os.environ.get("AZURE_PG_SSLMODE", "require"),
}

print(AZURE_OPENAI_DEPLOYMENT)
print(AZURE_EMBED_DEPLOYMENT)
print(AZURE_API_VERSION)

### Part 3.2: Tool 1 - `keyword_case_search` (BM25 full-text search)

> **Note:** This is the **first** tool in our agent's legal research pipeline. By the end of the notebook the agent will have five tools to choose from. We start small so you can see exactly what each piece does.

##### 🧠 *Technical Background Notes*

**Purpose:** perform fast, lexical (keyword-based) full-text search over Washington State case law opinions using the `pg_fts` PostgreSQL extension, which implements **BM25 ranking** (the same algorithm family used by Lucene, Elasticsearch, and Tantivy). This is excellent for surfacing canonical precedents when the user (or the agent) already knows the legal doctrine name, a citation, or distinctive terminology (e.g., "common enemy doctrine").

**Why BM25 instead of (or in addition to) vector search?** BM25 excels at exact-term and phrase matching. Vector search (coming up as Tool 2) excels at semantic / conceptual similarity. Using **both** gives the agent broader recall: keyword anchors the canonical cases, semantic expands to factually similar ones. This is the modern hybrid search pattern, and HorizonDB lets us do it inside a single database.

**How the `@tool` decorator works:** the `@tool` decorator from Agent Framework registers this Python function as a callable tool for the LLM agent. It inspects the function signature plus `Annotated[..., Field(description=...)]` metadata to build the JSON schema the agent sees when deciding whether to call this tool and what arguments to pass. In other words, the descriptions you write are the agent's user manual: write them well and the agent will use the tool correctly.

**What you'll see in the output:** a `print("keyword_case_search was called")` line confirms the tool fired, followed by the bound arguments. Later, the Gradio UI's "Tool Trace" panel keys off that exact string to surface tool calls to end users.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. This cell only **defines** the tool function, so the output is just a success check mark. We'll actually invoke it in the next cell.

In [ ]:
@tool(description=(
    "BM25 full-text search over Washington case law opinions. "
    "Supports phrase proximity ('common enemy'~3), boolean operators "
    "(AND, OR, NOT), and optional filters by court level and minimum decision date. "
    "Returns the top matching cases ranked by BM25 relevance score. "
    "Use this first to surface canonical precedents by doctrine name or citation."
))
def keyword_case_search(
    # The main search query. Tantivy/BM25 syntax supports:
    #   - Plain terms:  surface water drainage
    #   - Phrases:      "surface water"
    #   - Proximity:    "common enemy"~3   (within 3 tokens of each other)
    #   - Booleans:     AND, OR, NOT
    #   - Grouping:     (regrade OR grading)
    query: Annotated[str, Field(
        description=(
            "BM25/Tantivy-syntax query over case opinions. "
            "Examples: 'common enemy doctrine', "
            "'\"surface water\"~3 AND (regrade OR grading)', "
            "'qualified immunity NOT prison'."
        )
    )],
    # Optional metadata filter — restrict to one court tier. Useful when
    # the agent wants only binding precedent (Supreme Court) vs.
    # persuasive authority (Court of Appeals).
    court_level: Annotated[Optional[str], Field(
        default=None,
        description=(
            "Optional filter: 'Washington Supreme Court' "
            "or 'Washington Court of Appeals'. "
            "None = both courts."
        ),
    )] = None,
    # Optional metadata filter — exclude cases older than `min_year`.
    # Helpful for narrowing to modern doctrine while still allowing access
    # to historical canonical cases when min_year is None or very low.
    min_year: Annotated[Optional[int], Field(
        default=None,
        description="Optional filter: only return cases decided on or after this year."
    )] = None,
) -> str:
    # Diagnostic logging — printed to stdout so the Gradio UI's
    # "Tool Trace" panel can detect that this tool was invoked and display
    # its parameters. The trace parser keys off the literal string
    # "<tool_name> was called", so keep that line shape stable.
    print("keyword_case_search was called")
    print(f"  query={query!r}  court_level={court_level}  min_year={min_year}  limit=5")

    # Open a short-lived connection to Azure Database for PostgreSQL.
    # autocommit=True avoids needing an explicit COMMIT for SET statements.
    # Using a context manager ensures the connection + cursor are closed
    # cleanly even if an exception is raised mid-query.
    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
        # Initialize pg_fts for this session (temporary step during preview).
        cur.execute("SELECT pgfts.hello_pg_fts();")

        # Add the pgfts schema to the session search_path so we can call
        # fts_query() and fts_score() unqualified in the SELECT below.
        cur.execute("SET search_path = public, pgfts;")

        # ──────────────────────────────────────────────────────────────
        # Build the WHERE clause dynamically so optional filters
        # (court_level, min_year) only appear when provided. The first
        # predicate is the BM25 match against the pre-built FTS index
        # named 'idx_cases_fts' (created in notebook 1). Parameters are
        # passed via psycopg's parameter binding (%s) — never via string
        # concatenation — to keep the query safe from SQL injection.
        # ──────────────────────────────────────────────────────────────
        where = ["fts_query(%s, 'idx_cases_fts')"]
        params = [query]
        if court_level:
            where.append("court_level = %s")
            params.append(court_level)
        if min_year:
            # decision_date is a DATE column; compare against Jan 1 of the
            # requested year to act as a "year >= min_year" filter.
            where.append("decision_date >= %s")
            params.append(f"{min_year}-01-01")

        # Execute the search. fts_score(opinion) returns the BM25
        # relevance score for the matched row, which we surface to the
        # agent so it can reason about how strong each match is.
        # LIMIT 5 keeps the response small enough for the LLM context
        # window and bounds latency.
        cur.execute(
            f"""
            SELECT id, name, decision_date, court_level,
                   fts_score(opinion) AS score, opinion
            FROM cases
            WHERE {' AND '.join(where)}
            LIMIT 5;
            """,
            tuple(params),
        )
        rows = cur.fetchall()

    # Early return when the query matched nothing — keeps downstream
    # parsing (the regex that pulls case ids out of each line) simple.
    if not rows:
        return "No matches"

    # ──────────────────────────────────────────────────────────────────
    # Format results as a pipe-delimited text block. This shape matters:
    # downstream code (the parse_ids helper used by later test cells and
    # by Tool 3's input extraction) uses a regex like  ^(\d+)\s+\|  to
    # extract case ids from the first column. If you change the format,
    # update those parsers too.
    #
    # The opinion is truncated to 300 chars to keep the agent's context
    # window manageable while still giving it enough text to reason over.
    # Newlines in the opinion are flattened so each row stays on one
    # line — again, to keep the line-based parser working.
    # ──────────────────────────────────────────────────────────────────
    lines = []
    for case_id, name, decision_date, level, score, opinion in rows:
        snippet = (opinion or "").replace("\n", " ")[:300]
        lines.append(
            f"{case_id} | {decision_date} | {level} | score={score:.3f} | "
            f"{name}: {snippet}..."
        )
    return "\n".join(lines)


#### Part 3.2.1: Smoke-test Tool 1

> **Note:** Before we hand `keyword_case_search` to an LLM, let's call it directly as a regular Python function and confirm the plumbing works end-to-end against HorizonDB.

##### 🧠 *Technical Background Notes*

This is the simplest possible check: a plain Python function call, no agent involved. We pass a hand-written **BM25** query string and print the raw result. Things to look for in the output:

- **The query syntax itself**: `"surface water OR flooding OR drainage"` is BM25/Tantivy query syntax. The `OR` operator widens recall by matching any of the three terms. BM25 also supports phrase matches (`"surface water"`), proximity (`"common enemy"~3`), boolean logic (`AND`, `OR`, `NOT`), and grouping (`(regrade OR grading)`).
- **One row per match**, pipe-delimited: `case_id | decision_date | court_level | score=... | name: opinion snippet...`
- **The `score=` field** is the **BM25 relevance score** returned by `pg_fts`. Higher means a stronger lexical match (term frequency, inverse document frequency, and document-length normalization all factor in). It is *not* a probability or percentage, just a rank-order signal. Use it to compare matches within a single query.
- **Truncated opinions**: each opinion snippet is cut to 300 chars so the result stays small enough to feed into an LLM context window later.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. Confirm you see up to five pipe-delimited rows with a `score=` value on each. If you do, the tool is healthy and ready to be handed to an agent.

In [ ]:
print("// Test Call of keyword_case_search //")
print(keyword_case_search("surface water OR flooding OR drainage"))

#### Part 3.2.2: Assemble your first agent

> **Note:** Same tool, very different experience. Now an LLM gets to decide *when* and *how* to call `keyword_case_search`, based purely on a natural-language question.

##### 🧠 *Technical Background Notes*

So far we've just defined and unit-tested `keyword_case_search` as a plain Python function. In this cell we **promote** it into an actual agent: an LLM that autonomously decides **when** to call the tool, **what** arguments to pass, and **how** to weave the tool's output into a natural-language answer.

This is the "smallest interesting" version of the agent: one tool, one prompt. Later cells will add semantic search (Tool 2), the citation graph (Tool 3), entity extraction (Tool 4), and weather evidence (Tool 5), then reassemble the agent each time so you can see how additional tools change its behavior. It's like watching a Pokémon evolve, only with more SQL. 🐘

Two key pieces of Agent Framework glue to notice:

- **`OpenAIChatClient`**: despite the name, passing `azure_endpoint` routes calls to your **Azure OpenAI** deployment. The `model` argument is the *deployment name* you set in Azure AI Foundry, not the base model name like `gpt-4o`.
- **`client.as_agent(instructions=..., tools=[...])`**: wraps the chat client in an agent loop that advertises your `@tool`-decorated functions to the LLM, executes any tool calls the model makes, feeds results back into the conversation, and keeps looping until the model produces a final answer.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. Watch for the `keyword_case_search was called` line in the output: that's proof the LLM chose to call our tool, with arguments **it** picked from the natural-language question. Compare the agent's narrative answer to the raw rows from the smoke test above.

In [ ]:
# Build the underlying chat client.
#
# Despite the class name `OpenAIChatClient`, this is configured to talk
# to Azure OpenAI by passing `azure_endpoint`. The Agent Framework SDK
# routes requests to your Azure OpenAI deployment using:
#   - model:           the Azure OpenAI deployment name (NOT the base
#                      model name like "gpt-4o"). This is the name you
#                      gave the deployment in Azure AI Foundry / the
#                      Azure portal.
#   - azure_endpoint:  the resource endpoint, e.g.
#                      https://<your-resource>.openai.azure.com/
#   - api_key:         the resource key. In production prefer Azure AD /
#                      Managed Identity over a static key.
client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

# Turn the chat client into an Agent.
#
# `as_agent(...)` wraps the chat client in an agent loop that:
#   1. Sends the user message + the system `instructions` to the LLM.
#   2. Advertises the `tools` list to the LLM as callable functions
#      (the JSON schema is auto-generated from each @tool-decorated
#      function's signature and Annotated[..., Field(description=...)]
#      metadata — that's why those descriptions matter).
#   3. If the LLM responds with a tool call, the framework executes the
#      Python function, feeds the result back into the conversation,
#      and lets the model continue reasoning.
#   4. Repeats until the model produces a final natural-language answer.
#
# `instructions` is the system prompt. Keep it short and directive:
#   - Tells the model its role ("legal assistant").
#   - Tells it the desired output shape (case names + why relevant).
#   - "Run each tool only once" discourages the model from re-calling
#     the same tool with slightly tweaked arguments, which wastes
#     latency and token budget without improving recall.
agent = client.as_agent(
    instructions=(
        "You are a helpful legal assistant. Respond with the case names, and why each is relevant. Run every tool only once."
    ),
    tools=[keyword_case_search],
)

# The user question. Notice we do NOT pre-extract keywords or hand the
# tool a search string directly — that's the agent's job. The LLM will
# read this natural-language prompt, decide which terms to search for
# (e.g., "surface water", "drainage", "common enemy doctrine"), and
# call keyword_case_search with arguments of its own choosing.
user_query = (
    "I have a client in Seattle, WA whose property keeps flooding after the developer "
    "next door regraded their lot and the city redid the drainage on the "
    "road."
)

# Run the agent. `agent.run(...)` is async, so we `await` it. Because
# we called `nest_asyncio.apply()` at the top of the notebook, awaiting
# at the top level of a cell works inside Jupyter.
#
# While this runs, watch the cell output: the `print(...)` lines inside
# keyword_case_search (e.g., "keyword_case_search was called") will
# appear, which is how you confirm the agent actually invoked the tool
# rather than hallucinating an answer from prior knowledge.
print("// Functions the Agent Called: //")
result = await agent.run(user_query)

# `result.text` is the agent's final natural-language answer (after any
# tool calls have been folded in). We render it as Markdown so case
# names, bullets, and formatting display nicely in the notebook.
print("")
print("// Agent Response: //")
Markdown(result.text)


### Part 3.3: Tool 2 - `semantic_case_search` (vector search with DiskANN advanced filtering)

> **Note:** This is the **second** tool in our agent's pipeline. Where Tool 1 matched on the *exact words* you typed, Tool 2 matches on the *meaning* behind them. Together they give the agent the best of both worlds.

##### 🧠 *Technical Background Notes*

**Purpose:** find cases that are *conceptually* similar to a natural-language fact pattern, even when the user does not know the legal doctrine name or canonical citation. Tool 1 (`keyword_case_search`) is great when you already know the right terms (e.g., "common enemy doctrine"); Tool 2 shines when you only know the situation (e.g., "my client's lot floods after a neighbor regraded their land"). Under the hood, the query is embedded by Azure OpenAI's embedding model and matched against the `cases.opinion_embedding` column in HorizonDB using `pgvector` cosine similarity.

**Pipeline role and how results combine with Tool 1:**

- Tool 1 returns up to 5 cases by BM25 keyword relevance.
- Tool 2 returns up to 5 cases by vector similarity, **independently** of Tool 1 (this function does not look at or merge Tool 1's results).
- The **caller** (the agent, or the test cells below) takes the **union** of those id sets, up to 10 unique cases, and passes them to Tool 3 as `input_case_ids` for graph expansion. Classic hybrid retrieval, all inside one Postgres.

**🌟 DiskANN Advanced Filtering, always on:** this lab is specifically about showing off HorizonDB's `pg_diskann` extension and its **advanced filtering** capability, which pushes `WHERE` predicates **into** the index traversal rather than applying them as a post-filter. The naive approach (post-filter) can blow up: the index returns the top-N by similarity, *then* the filter throws most of them away, leaving you with too few results or none at all. Advanced filtering avoids that by honoring the predicate while the index walks the graph. To make sure the feature is exercised on every call, the session GUCs are set unconditionally in the code below: there is no opt-out flag.

**Why `court_level` and `min_year` are required (not optional):** these are deliberately **required** parameters so the LLM always supplies values. That guarantees the `WHERE` clause has predicates for DiskANN advanced filtering to act on, so the lab consistently demonstrates filtered vector search rather than degenerating into an unfiltered top-k.

- `court_level` accepts `"Both"` as the explicit "do not filter by court" value, so the model never has the ambiguous option of just leaving it out.
- `min_year` accepts a low year like `1900` to mean "no real cutoff", while still giving DiskANN's advanced filtering a predicate to push down.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. This cell only **defines** the tool function, so the output is just a success check mark. We'll invoke it in the next cell.

In [ ]:
# We validate against this set so a typo from the LLM produces a clean error
# message instead of silently returning zero rows.
VALID_COURT_LEVELS = {"Washington Supreme Court", "Washington Court of Appeals", "Both"}

@tool(description=(
    "Semantic vector search over Washington case opinions using DiskANN with "
    "Advanced Filtering enabled (filters are pushed into the index traversal). "
    "Returns the top 5 cases ranked by cosine similarity to the query embedding. "
    "Both court_level and min_year are REQUIRED — pass court_level='Both' to search "
    "across both courts, and pass min_year=1900 if you do not want a date cutoff. "
    "This tool runs independently of keyword_case_search; the caller is expected to "
    "take the UNION of ids from both tools before passing them to precedent_graph_search."
))
def semantic_case_search(
    # The natural-language query. Unlike Tool 1, no special syntax —
    # the embedding model captures meaning, not literal tokens. So
    # "developer regraded lot causing flooding" works as well as a
    # carefully crafted boolean expression would in BM25.
    query_text: Annotated[str, Field(
        description="Natural language query describing the fact pattern or legal issue."
    )],
    # REQUIRED court filter. The model must pick one of the three
    # values. "Both" is the explicit way to skip filtering by court so
    # the model never has to omit the argument.
    court_level: Annotated[Literal["Washington Supreme Court", "Washington Court of Appeals", "Both"], Field(
        description=(
            "REQUIRED court filter. Use 'Washington Supreme Court' for binding precedent only, "
            "'Washington Court of Appeals' for that court only, or 'Both' to search across both courts."
        ),
    )],
    # REQUIRED year-floor filter. The model must pass an integer; use
    # 1900 (or earlier) to effectively disable the cutoff while still
    # giving DiskANN's advanced filtering a predicate to work with.
    min_year: Annotated[int, Field(
        description=(
            "REQUIRED minimum decision year. Only cases decided on or after Jan 1 of this year "
            "are returned. Pass 1900 to effectively disable the cutoff."
        )
    )],
) -> str:
    # Diagnostic trace line — the Gradio UI's Tool Trace panel keys off
    # the literal "<tool_name> was called" string, so keep that shape.
    print("semantic_case_search was called")
    print(
        f"  query_text={query_text!r}  court_level={court_level}  "
        f"min_year={min_year}  limit=5  advanced_filtering=ALWAYS_ON"
    )

    # Validate against the whitelist. Returning early with a clear
    # message helps the LLM self-correct on its next turn.
    if court_level not in VALID_COURT_LEVELS:
        return f"Invalid court_level {court_level!r}. Use one of: {sorted(VALID_COURT_LEVELS)}"

    # Short-lived autocommit connection — SET statements affect only
    # this session, so a fresh connection per call gives us a clean
    # GUC state every time.
    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
        # ──────────────────────────────────────────────────────────────
        # DiskANN advanced-filtering GUCs (session-level) — ALWAYS ON.
        # These tell the DiskANN index to evaluate the WHERE predicates
        # DURING graph traversal rather than as a post-filter. The
        # values below are sensible defaults for this lab; tune per
        # workload in production.
        #
        #   enable_filter_hook    – master switch for advanced filtering.
        #   selectivity_min/_threshold – when to engage filtered traversal
        #         based on estimated predicate selectivity (fraction of
        #         rows expected to match). 0.0..1.0 means "always try".
        #   filtering_beta        – tradeoff between recall and latency
        #         when navigating the filtered subgraph. Higher = more
        #         exploration = better recall, slightly slower.
        #   l_value_is            – search list size for the in-memory
        #         portion of the index. Larger = better recall, slower.
        # ──────────────────────────────────────────────────────────────
        cur.execute("SET diskann.enable_filter_hook = 'true';")
        cur.execute("SET diskann.selectivity_min = '0.0';")
        cur.execute("SET diskann.selectivity_threshold = '1.0';")
        cur.execute("SET diskann.filtering_beta = 0.85;")
        cur.execute("SET diskann.l_value_is = 300;")

        # ──────────────────────────────────────────────────────────────
        # Build the WHERE clause. We use NAMED placeholders (%(name)s)
        # so we can pass a single params dict.
        #
        # `opinions_vector IS NOT NULL` excludes any case where the
        # embedding wasn't generated (defensive — shouldn't happen in
        # this lab, but cheap insurance).
        #
        # court_level filter is added unless the model passed "Both"
        # (the explicit "no court filter" sentinel). min_year is always
        # applied — pass 1900 for "no cutoff".
        # ──────────────────────────────────────────────────────────────
        where = [
            "opinions_vector IS NOT NULL",
            "decision_date >= make_date(%(min_year)s, 1, 1)",
        ]
        params = {
            "q": query_text,
            "lim": 5,
            "min_year": min_year,
            # The embedding deployment name (e.g. "text-embedding-3-small")
            # is what azure_openai.create_embeddings() needs to know which
            # Azure OpenAI deployment to call. The endpoint + key were
            # configured back in Notebook 1 via azure_ai.set_setting().
            "embed_deployment": AZURE_EMBED_DEPLOYMENT,
        }

        if court_level != "Both":
            where.append("court_level = %(court_level)s")
            params["court_level"] = court_level

        # ------------------------------------------------------------------
        # Embedding call: pick ONE of the two approaches below.
        #
        # (A) AI Model Management (managed alias) -- COMMENTED OUT for now.
        #     Uses azure_ai.create_embeddings() with a configured model alias
        #     (e.g. 'default-embedding') that maps to a backing deployment via
        #     AI Model Management or BYOM model_registry.model_add. Leave this
        #     here so we can switch back when managed models are enabled.
        #
        #     embed_sql = (
        #         "azure_ai.create_embeddings('default-embedding', %(q)s)::vector"
        #     )
        #
        # (B) Traditional azure_ai extension (ACTIVE).
        #     Uses azure_openai.create_embeddings(<deployment>, <text>) which
        #     reads the endpoint + subscription key set in Notebook 1 via
        #     azure_ai.set_setting('azure_openai.endpoint'/'subscription_key').
        #     The deployment name comes straight from AZURE_EMBED_DEPLOYMENT.
        # ------------------------------------------------------------------
        # `::vector` casts the JSONB array returned by the function into
        # pgvector's `vector` type so the `<=>` operator below works.
        embed_sql = (
            "azure_openai.create_embeddings(%(embed_deployment)s, %(q)s)::vector"
        )

        # ──────────────────────────────────────────────────────────────
        # The core semantic-search query.
        #
        #   `opinions_vector <=> <query_vector>`  – cosine DISTANCE
        #       (NOT similarity). Smaller = more similar. 0.0 is a
        #       perfect match; 2.0 is opposite. Other pgvector operators:
        #         <->  L2 distance
        #         <#>  negative inner product
        #
        #   ORDER BY distance ASC + LIMIT 5  – this is the pattern the
        #   DiskANN planner recognizes to engage the index. Without an
        #   ORDER BY on the distance expression it would fall back to a
        #   sequential scan, which is dramatically slower on large
        #   corpora.
        # ──────────────────────────────────────────────────────────────
        cur.execute(
            f"""
            SELECT id, name, decision_date, court_level,
                   opinions_vector <=> {embed_sql} AS distance,
                   opinion
            FROM public.cases
            WHERE {' AND '.join(where)}
            ORDER BY distance ASC
            LIMIT %(lim)s;
            """,
            params,
        )
        rows = cur.fetchall()

    # Nothing matched.
    if not rows:
        return "No matches"

    # ──────────────────────────────────────────────────────────────────
    # Format the response. Same pipe-delimited shape as Tool 1 so the
    # `parse_ids` helper used in later cells can extract case ids from
    # either tool's output with the same regex (^(\d+)\s+\|).
    # Snippets are flattened to one line and truncated to 300 chars to
    # control LLM context size.
    #
    # NOTE: This output contains only the 5 vector-search hits. To feed
    # Tool 3, take the UNION of these ids with Tool 1's ids — see the
    # test cell below for the pattern.
    # ──────────────────────────────────────────────────────────────────
    lines = []
    for case_id, name, decision_date, level, dist, opinion in rows:
        snippet = (opinion or "").replace("\n", " ")[:300]
        dist_str = "n/a" if dist is None else f"{dist:.4f}"
        lines.append(
            f"{case_id} | {decision_date} | {level} | distance={dist_str} | "
            f"{name}: {snippet}..."
        )

    return "\n".join(lines)


#### Part 3.3.1: Smoke-test Tool 1 and Tool 2 together (no agent)

> **Note:** Before we add Tool 2 to the agent, let's call both tools directly back-to-back and see how their results combine. This is the "human-in-the-loop" version of what the agent will soon do on its own.

##### 🧠 *Technical Background Notes*

This cell exercises the hybrid retrieval pattern by hand, in three steps:

1. **Keyword anchors (Tool 1, BM25)**: a precise BM25/Tantivy query against `pg_fts` to lock onto canonical cases that mention the exact doctrine. Notice the proximity operator `'common enemy'~3` (the two terms within 3 tokens of each other) combined with an `AND (... OR ...)` group: this is the kind of expression an experienced legal researcher would write.
1. **Semantic expansion (Tool 2, `pgvector` + DiskANN)**: a natural-language fact pattern is embedded by Azure OpenAI and matched by cosine similarity. Required filters (`court_level="Both"`, `min_year=1900`) are passed so DiskANN's **advanced filtering** runs even when we don't actually want to narrow the results.
1. **Union of ids**: a tiny regex picks the leading `case_id` off each pipe-delimited row from both tools, then `set(kw_ids) | set(sem_ids)` produces the de-duplicated input list. Up to 10 unique cases will eventually be handed to Tool 3 (the citation graph) for one-hop expansion.

##### 👀 *What you'll see in the output*

When you run the cell, you'll get three labeled blocks separated by blank lines:

- **`// Step 1: Keyword anchors (BM25) ... //`** followed by up to 5 pipe-delimited rows: `case_id | decision_date | court_level | score=... | name: opinion snippet...`. The `score=` value is the BM25 relevance from `pg_fts`. Higher = stronger lexical match.
- **`// Step 2: Semantic expansion (DiskANN, Advanced Filtering on) ... //`** followed by another set of rows. Same row format, but now the score is a **cosine similarity** value from `pgvector` (closer to 1.0 = more semantically similar to the embedded fact pattern). The trace line `semantic_case_search was called` confirms the tool fired and shows the `advanced_filtering=ALWAYS_ON` indicator.
- **`// Step 3 preview: union ... //`** followed by three lines:
    - `Tool 1 ids (N): [...]` - the ids extracted from BM25 results
    - `Tool 2 ids (N): [...]` - the ids extracted from vector results
    - `Union  ids (N): [...]` - the de-duplicated combined list, which becomes Tool 3's `input_case_ids`

The cool part: **expect overlap, but not total overlap**. Cases that show up in both lists are strong "this is definitely on point" candidates. Cases that show up in only one list are the unique value each retrieval style adds. That diversity is exactly why hybrid search beats either approach alone.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. Compare the two id lists. How many cases overlap? How many are unique to each tool? That gap is what hybrid retrieval buys you.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Step 1: Tool 1 (BM25 keyword search)
# ─────────────────────────────────────────────────────────────────────
# We hand-craft a precise BM25/Tantivy query that an experienced legal
# researcher might write. Notable operators:
#   - 'common enemy'~3   : the two terms must appear within 3 tokens
#   - AND ( ... OR ... ) : require the doctrine phrase AND at least one
#                          of the related drainage terms
# Filtering to "Washington Supreme Court" keeps results to binding
# precedent only; min_year=1900 effectively disables the date floor.
print("// Step 1: Keyword anchors (BM25) - returns up to 5 cases //")
kw = keyword_case_search(
    "'common enemy'~3 AND (surface water OR drainage OR stormwater)",
    court_level="Washington Supreme Court",
    min_year=1900,
)
print(kw)

# Each result row from our tools starts with "<case_id> | ..." (pipe
# delimited). This regex pulls just the leading integer id off each
# line so we can build a de-duplicated input set for Tool 3 later.
kw_ids = []
for line in kw.splitlines():
    m = re.match(r"^(\d+)\s+\|", line)
    if m:
        kw_ids.append(int(m.group(1)))

print("")

# ─────────────────────────────────────────────────────────────────────
# Step 2: Tool 2 (semantic vector search via pgvector + DiskANN)
# ─────────────────────────────────────────────────────────────────────
# This time we pass a natural-language fact pattern. Tool 2 will embed
# it with Azure OpenAI and run a cosine-similarity search against the
# cases.opinion_embedding column. court_level and min_year are REQUIRED
# args, so we use the "no-op" sentinel values:
#   - court_level="Both"  -> do not filter by court
#   - min_year=1900       -> effectively no date cutoff
# Even when "no-op", these values still give DiskANN's advanced
# filtering a predicate to honor during index traversal.
print("// Step 2: Semantic expansion (DiskANN, Advanced Filtering on) - returns up to 5 cases //")
flagship_prompt = (
    "I have a client whose property keeps flooding after the developer next door "
    "regraded their lot and the city redid the drainage on the road."
)
sem = semantic_case_search(
    flagship_prompt,
    court_level="Both",   # "Both" = search across both courts (required arg, no None allowed)
    min_year=1900,        # 1900 = effectively no cutoff (required arg)
)
print(sem)

# Same regex parser as Step 1, applied to Tool 2's output to pull its
# up-to-5 case ids.
sem_ids = []
for line in sem.splitlines():
    m = re.match(r"^(\d+)\s+\|", line)
    if m:
        sem_ids.append(int(m.group(1)))

print("")

# ─────────────────────────────────────────────────────────────────────
# Step 3: Build the input set for Tool 3 (citation-graph expansion)
# ─────────────────────────────────────────────────────────────────────
# Tool 3 will consume the UNION of ids from Tool 1 + Tool 2 (up to 10
# unique cases). The set() | set() expression is Python's set-union
# operator, which automatically dedupes any case that was returned by
# both tools — those are the strongest "this is definitely on point"
# candidates. sorted(...) just gives stable output for readability.
print("// Step 3 preview: union of Tool 1 + Tool 2 ids -> inputs for Tool 3 //")
input_ids = sorted(set(kw_ids) | set(sem_ids))
print(f"Tool 1 ids ({len(kw_ids)}): {kw_ids}")
print(f"Tool 2 ids ({len(sem_ids)}): {sem_ids}")
print(f"Union  ids ({len(input_ids)}): {input_ids}")


#### Part 3.3.2: Re-assemble the agent with Tool 1 **and** Tool 2

> **Note:** Same agent pattern as Part 3.2.2, but now the LLM has **two** retrieval tools to pick from. Watch how it chooses.

##### 🧠 *Technical Background Notes*

In Part 3.2.2 the agent had a single tool (`keyword_case_search`), so every question funneled through BM25. Now we hand it both `keyword_case_search` and `semantic_case_search` and let the model decide which to call, in what order, and with what arguments. This is the whole point of an agent loop: tool selection is data-driven, not hard-coded.

A few things to notice:

- **Fresh `OpenAIChatClient` + `as_agent(...)`**: we rebuild the agent from scratch each time we add tools. Agent Framework's `as_agent()` snapshots the `tools=[...]` list at construction time, so re-creating the agent is the cleanest way to expose new capabilities to the LLM.
- **Same prompt as the smoke test**: we deliberately reuse the Seattle flooding scenario from Part 3.3.1 so you can compare the agent's narrative answer against the raw BM25 + vector rows you just printed. The underlying retrieval is identical: only the orchestration layer changes.
- **"Run each tool only once"** in the instructions is a gentle nudge to keep the trace readable for the lab. In production you'd usually let the agent re-query as needed.
- **Tool-call trace**: the `print("keyword_case_search was called")` / `print("semantic_case_search was called")` lines inside each tool fire as the agent invokes them, so the `// Functions the Agent Called: //` block becomes a live trace of the LLM's decisions.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. In the output, confirm that **both** tools are called (you should see both `keyword_case_search was called` and `semantic_case_search was called`). If the agent only calls one, that's a sign the prompt or instructions need tightening.

1. Read the rendered Markdown answer and compare it to the raw rows from Part 3.3.1. Notice how the agent now cites case names and explains *why* each is relevant, instead of dumping pipe-delimited rows.


In [ ]:
# Step 1: Build a fresh Azure OpenAI chat client.
# OpenAIChatClient + azure_endpoint routes through your Azure OpenAI deployment.
# `model` here is the *deployment name* from Azure AI Foundry, not the base model id.
client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

# Step 2: Wrap the chat client in an agent that knows about BOTH retrieval tools.
# `as_agent()` snapshots the tools list at construction time, so we rebuild the
# agent each time we add a new tool. The instructions are the agent's system
# prompt: keep them short and behavioral ("respond with...", "run each tool only once").
agent = client.as_agent(
    instructions=(
        "You are a helpful legal assistant. Respond with the case names, why each is relevant. Run each tool only once."
    ),
    tools=[
        keyword_case_search,
        semantic_case_search
    ],
)

# Step 3: Same Seattle flooding scenario as the Part 3.3.1 smoke test, so you can
# compare the agent's narrative answer against the raw rows we printed earlier.
user_query = (
    "I have a client in Seattle, WA whose property keeps flooding after the developer "
    "next door regraded their lot and the city redid the drainage on the "
    "road."
)

# Step 4: Run the agent. Under the hood Agent Framework will:
#   (a) send the prompt + tool schemas to Azure OpenAI,
#   (b) execute any tool calls the model requests (printing the "was called" lines),
#   (c) feed tool outputs back into the conversation,
#   (d) loop until the model returns a final natural-language answer.
print("// Functions the Agent Called: //")
result = await agent.run(user_query)

# Step 5: Render the agent's final answer as Markdown so case names, bullets,
# and emphasis show up nicely in the notebook output.
print("")
print("// Agent Response: //")
Markdown(result.text)


### Part 3.4: Tool 3 - `precedent_graph_search` (citation graph traversal with Apache AGE)

> **Note:** This is the **third** tool in our agent's pipeline. Tools 1 and 2 *find* candidate cases; Tool 3 *expands* that set by walking the citation graph to surface the binding precedents and influential cases connected to them.

##### 🧠 *Technical Background Notes*

**Purpose:** take the union of case ids returned by `keyword_case_search` (Tool 1) and `semantic_case_search` (Tool 2) as `input_case_ids`, then traverse the Washington case citation graph stored in **Apache AGE** to return the top related precedents. This is how the agent moves from "cases that look relevant" to "cases that the legal community has actually cited as authority on this question".

**🌟 Apache AGE inside HorizonDB:** AGE is the PostgreSQL extension that adds an OpenCypher property-graph engine *into* the same database that already holds your relational rows and vector embeddings. The graph is named `case_graph`, nodes are `:case` with a `case_id` property, and edges are `(a)-[:REF]->(b)` meaning **"a cites b"**. Because AGE lives next to the `public.cases` table, we can do a Cypher traversal **and** a relational join in the same connection: no ETL, no second datastore, no graph-database operations team.

**The traversal pattern:**

- **2 hops, both directions.** We run two Cypher queries: one follows outgoing `REF` edges (input -> authority it cites), the other follows incoming edges (later case -> input). Aggregating both gives a complete picture of the line of authority around the input cluster. 1 hop is usually too narrow; 3 hops pulls in weakly-related cases and explodes the result size.
- **Path counting as a centrality proxy.** A case reachable from *many* inputs via *many* distinct multi-hop paths is, by definition, sitting closer to the center of the citation cluster. We just `count(*)` the paths per `related_id` and treat that as the base relevance signal.
- **Skip the inputs themselves.** Tools 1 and 2 already returned those; here we only want *expansions*.

**Ranking - `score = paths + court_weight`:**

- `paths` is the centrality proxy above.
- `court_weight` adds **+2** for Washington Supreme Court (binding precedent) and **0** for everything else. This is a coarse boost: when path counts are otherwise comparable, binding precedent always outranks persuasive Court of Appeals authority. The boost is **always on**: there is no opt-out, because the lab is showing the realistic legal-research behavior.
- Ties on `score` are broken by raw `paths`. We keep the **top 10** so Tool 4 (extract) and the LLM context window do not get flooded.

**Hybrid graph + relational in one query path:** after the Cypher traversal returns `related_id`s, we immediately `SELECT id, name, decision_date, court_level, opinion FROM public.cases WHERE id = ANY(%s)` to enrich each hit with metadata and an opinion snippet. One Postgres connection, two query languages, zero data movement.

**Output format:** the tool returns a structured text blob with a `RELATED_CASE_IDS: ...` header line (parseable handoff for Tool 4) followed by human-readable rows of `case_id | date | court_level | paths=N | score=N | name: snippet...`. The `print("precedent_graph_search was called")` line is what the Gradio "Tool Trace" panel keys off later.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. This cell only **defines** the tool function, so the output is just a success check mark. We'll invoke it in the next cell against the `input_ids` produced by Tools 1 and 2.


In [ ]:
@tool(description=(
    "Traverse the Washington case citation graph stored in Apache AGE (graph name: case_graph). "
    "Edges are (a)-[:REF]->(b) meaning 'a cites b'. "
    "Given input case ids (from keyword_case_search and semantic_case_search), "
    "returns the TOP 10 most relevant related precedent cases (by centrality score) via a "
    "2-hop bidirectional citation traversal (follows both 'cites' and 'cited-by' edges), "
    "enriched with metadata from the relational cases table. "
    "Washington Supreme Court cases are always boosted in the ranking as binding precedent."
))
def precedent_graph_search(
    input_case_ids: Annotated[List[int], Field(
        description=(
            "List of input case ids from Tool 1 and/or Tool 2. "
            "Pass the union of ids from both tools for best coverage."
        )
    )],
) -> str:
    # Cap the number of related cases returned, so Tool 4 (extract) doesn't
    # get flooded with low-relevance hits. We rank by `score` (see below)
    # and take the top N.
    TOP_K = 10

    # Court-weight boost for Washington Supreme Court (binding precedent).
    # Always applied — there is no opt-out, because binding precedent
    # should always outrank persuasive Court of Appeals authority when
    # path counts are otherwise comparable.
    SUPREME_COURT_BOOST = 2

    # Fixed traversal parameters — kept inside the function so the LLM
    # doesn't have to think about them and the lab demo stays consistent.
    #
    #   HOPS = 2  → up to cite-of-cite expansion. 1 would be direct
    #               citations only (often too narrow). 3 starts pulling
    #               in weakly-related cases and dramatically increases
    #               result size.
    #   DIRECTION = "both" → follow both outgoing edges (input cites prior
    #               authority) AND incoming edges (later cases cite the
    #               input). The union gives the most complete picture of
    #               the line of authority around the input cluster.
    HOPS = 2

    print("precedent_graph_search was called")
    print(f"  inputs={len(input_case_ids)}  direction=both  hops={HOPS}  top_k={TOP_K}  supreme_boost={SUPREME_COURT_BOOST}")

    if not input_case_ids:
        return "No input_case_ids provided."

    # Graph stores case_id as text property, created from temp_cases.data->>'id'
    # Your helper functions store it as text, so we compare using string literals.
    input_literals = ", ".join([f"'{int(i)}'" for i in sorted(set(input_case_ids))])

    # Build Cypher fragments for both directions of the REF edge.
    # REF direction: (a)-[:REF]->(b) means "a cites b"
    #   - "cites":    input -> authority          (outgoing from input)
    #   - "cited_by": later_case -> input         (incoming to input,
    #                                              traversed by matching
    #                                              (t)-[:REF*]->(s))
    # We always run both and aggregate path counts together.
    cypher_queries = [
        f"""
            SELECT * FROM cypher('case_graph', $$
                MATCH (s:case)-[:REF*1..{HOPS}]->(t:case)
                WHERE s.case_id IN [{input_literals}]
                RETURN t.case_id AS related_id, count(*) AS paths
            $$) AS (related_id agtype, paths agtype);
        """,
        f"""
            SELECT * FROM cypher('case_graph', $$
                MATCH (t:case)-[:REF*1..{HOPS}]->(s:case)
                WHERE s.case_id IN [{input_literals}]
                RETURN t.case_id AS related_id, count(*) AS paths
            $$) AS (related_id agtype, paths agtype);
        """,
    ]

    # Execute graph traversal and aggregate path counts.
    related_paths = {}  # related_id(int) -> paths(int)
    try:
        with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
            # AGE session setup            
            cur.execute("SET search_path = public, ag_catalog, \"$user\";")

            for q in cypher_queries:
                cur.execute(q)
                for related_id_ag, paths_ag in cur.fetchall():
                    # AGE returns agtype; psycopg gives it as Python str like '"123"' or '123'
                    rid_str = str(related_id_ag).strip('"')
                    if not rid_str.isdigit():
                        continue
                    rid = int(rid_str)

                    # Skip inputs in the related set; we only want expansions
                    if rid in input_case_ids:
                        continue

                    p = int(str(paths_ag).strip('"')) if str(paths_ag).strip('"').isdigit() else 1
                    related_paths[rid] = related_paths.get(rid, 0) + p

    except Exception as e:
        return f"Graph query failed. Is case_graph created and populated? Error: {e}"

    if not related_paths:
        return "No related cases found from the citation graph for the provided inputs."

    related_ids = list(related_paths.keys())

    # Enrich from relational table so Tool 4 can immediately extract from opinion text.
    # Also compute a simple ranking score: paths (centrality proxy) + court weighting.
    try:
        with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
            cur.execute(
                """
                SELECT id, name, decision_date, court_level, opinion
                FROM public.cases
                WHERE id = ANY(%s);
                """,
                (related_ids,)
            )
            rows = cur.fetchall()
    except Exception as e:
        return f"Failed to fetch case metadata from relational table. Error: {e}"

    # Court weighting: WA Supreme Court always gets the binding-precedent
    # boost; Court of Appeals (and anything else) gets 0.
    def court_weight(level: str) -> int:
        if level == "Washington Supreme Court":
            return SUPREME_COURT_BOOST
        return 0

    # ──────────────────────────────────────────────────────────────────
    # Score = paths + court_weight
    #
    #   paths        = count of distinct multi-hop citation paths from
    #                  any input to this case (a centrality proxy — more
    #                  paths = the case sits closer to the input cluster
    #                  in the citation graph).
    #   court_weight = +2 if Washington Supreme Court, else 0. Coarse
    #                  boost for binding precedent over Court of Appeals.
    #                  Always applied (no opt-out).
    #
    # Ties on `score` are broken by raw `paths`.
    # ──────────────────────────────────────────────────────────────────
    ranked = []
    for case_id, name, decision_date, court_level, opinion in rows:
        paths = related_paths.get(case_id, 0)
        score = paths + court_weight(court_level)
        snippet = (opinion or "").replace("\n", " ")[:240]
        ranked.append((score, paths, case_id, decision_date, court_level, name, snippet))

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)

    # Keep only the top-K highest-scoring related cases. This bounds the
    # payload size handed off to Tool 4 (extract) and to the LLM context
    # window, while still keeping the most central / binding precedents.
    total_related = len(ranked)
    ranked = ranked[:TOP_K]

    print(f"  related_count_total={total_related}  returned_top_k={len(ranked)}")

    # Return a parseable, stable text format:
    # - first section: ids only (for Tool 4 handoff) — already trimmed to top-K
    # - second section: human-readable justification
    top_ids = [str(r[2]) for r in ranked]
    lines = []
    lines.append("RELATED_CASE_IDS: " + ", ".join(top_ids))
    lines.append("")
    lines.append(f"Related cases (top {len(ranked)} of {total_related}, ranked by score):")
    for score, paths, case_id, decision_date, court_level, name, snippet in ranked:
        lines.append(
            f"{case_id} | {decision_date} | {court_level} | paths={paths} | score={score} | {name}: {snippet}..."
        )

    return "\n".join(lines)


#### Part 3.4.1: Smoke-test Tool 1, Tool 2 and Tool 3 together (no agent)

> **Note:** Before we hand Tool 3 to the agent, let's chain all three tools by hand. This is the "human-in-the-loop" version of the full retrieval pipeline: keyword + semantic + graph expansion.

###### 🧠 *Technical Background Notes*

This cell exercises the end-to-end retrieval pattern in three steps:

1. **Keyword anchors (Tool 1, BM25)**: same `'common enemy'~3 AND (surface water OR drainage OR stormwater)` query as Part 3.3.1, narrowed to `court_level="Washington Supreme Court"` because we want binding-precedent anchors to feed the graph.
1. **Semantic expansion (Tool 2, `pgvector` + DiskANN)**: same Seattle flooding fact pattern as Part 3.3.1, with `court_level="Both"` and `min_year=1900` so DiskANN's advanced filtering runs but does not actually narrow results.
1. **Graph traversal (Tool 3, Apache AGE)**: a tiny `parse_ids()` regex peels the leading `case_id` off each pipe-delimited row from Tools 1 and 2, takes the **union**, and passes it as `input_case_ids` to `precedent_graph_search`. The graph then expands those inputs into related precedents via 2-hop bidirectional `[:REF]` traversal.

This is the hybrid retrieval + graph expansion flow the agent will run on its own in Part 3.4.2: doing it by hand first makes the agent's tool-call trace much easier to read later.

##### 👀 *What you'll see in the output*

When you run the cell, you'll get three labeled blocks:

- **Tool 1 trace**: `keyword_case_search was called` plus the bound arguments, followed by up to 5 pipe-delimited BM25 rows (`case_id | decision_date | court_level | score=... | name: snippet...`).
- **Tool 2 trace**: `semantic_case_search was called` with the `advanced_filtering=ALWAYS_ON` indicator, followed by up to 5 cosine-similarity rows in the same format.
- **`Input ids: [...]`**: the de-duplicated union of ids from Tools 1 and 2. This is what gets passed to Tool 3 as `input_case_ids`.
- **`// Tool 3: Graph traversal //`** followed by:
    - `precedent_graph_search was called`
    - A parameters line: `inputs=N  direction=both  hops=2  top_k=10  supreme_boost=2`
    - A counts line: `related_count_total=X  returned_top_k=Y` (X is everything the graph found, Y is what survived the top-10 cap)
    - `RELATED_CASE_IDS: id1, id2, ...` - the handoff header that Tool 4 (coming up) will key off
    - A `Related cases (top N of M, ranked by score):` table where each row is `case_id | decision_date | court_level | paths=N | score=N | name: snippet...`

The interesting part:

- **None of the `Input ids` should appear in the `RELATED_CASE_IDS` row.** The graph deliberately skips the inputs themselves: we want *expansions*, not the cases we already had.
- **Washington Supreme Court rows should float to the top.** Look for `score = paths + 2` on those rows: that's the binding-precedent boost in action. A Court of Appeals case with the same `paths` value will sit lower because its `court_weight` is 0.
- **`paths` ≈ centrality.** A case with `paths=8` is reachable from your input cluster through 8 distinct multi-hop citation chains: that case is *highly* connected to the line of authority you started with.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. Inspect the Tool 3 output: are the top rows Washington Supreme Court cases? Confirm the `score = paths + supreme_boost` math by spot-checking a row or two.

1. Compare `related_count_total` against `returned_top_k`. If `total` is much larger than 10, that's how much the graph *could* have returned: the top-K cap is what keeps Tool 4 and the LLM context window happy.


In [ ]:
# Helper: extract case ids from a tool's text output.
#
# Tools 1 and 2 both return rows shaped like:
#     "12345 | 1953-04-02 | Washington Supreme Court | score=... | Name: snippet..."
# A tiny regex grabs the leading integer on each row so we can build the
# de-duplicated input set that Tool 3 (the citation graph) expects.
def parse_ids(tool_output: str) -> list[int]:
    ids = []
    for line in tool_output.splitlines():
        m = re.match(r"^(\d+)\s+\|", line)
        if m:
            ids.append(int(m.group(1)))
    return ids

# Step 1: Tool 1 (BM25 keyword search).
# We narrow to Washington Supreme Court because we want binding-precedent
# anchors to seed the graph traversal. The proximity operator
# 'common enemy'~3 keeps the two words within 3 tokens of each other.
kw_out = keyword_case_search(
    "'common enemy'~3 AND (surface water OR drainage OR stormwater)",
    court_level="Washington Supreme Court",
    min_year=1900,
)

# Step 2: Tool 2 (semantic / vector search with DiskANN advanced filtering).
# Natural-language fact pattern instead of keywords. "Both" courts and
# min_year=1900 are passed explicitly so DiskANN's advanced filtering
# always has predicates to push down (the lab demo is meant to exercise
# that code path on every call).
sem_out = semantic_case_search(
    "developer regraded lot causing flooding; city changed road drainage",
    court_level="Both",   # "Both" = search across both courts (required arg)
    min_year=1900,        # 1900 = effectively no cutoff (required arg)
)

# Step 3: Build the input set for Tool 3.
# Union of ids from Tools 1 + 2, de-duplicated and sorted for a stable
# trace. This list is exactly what the agent will compute on its own
# in Part 3.4.2 when it gets all three tools at once.
input_ids = sorted(set(parse_ids(kw_out) + parse_ids(sem_out)))
print("Input ids:", input_ids)

# Step 4: Tool 3 (Apache AGE citation graph traversal).
# Expands the input cluster via 2-hop bidirectional [:REF] edges and
# returns the top 10 related precedents, ranked by paths + court boost.
# Watch for: precedent_graph_search trace line, the parameters echo,
# related_count_total vs returned_top_k, and the RELATED_CASE_IDS header
# (which is the parseable handoff for Tool 4 later in the notebook).
print("")
print("// Tool 3: Graph traversal //")
graph_out = precedent_graph_search(input_case_ids=input_ids)
print(graph_out)


#### Part 3.4.2: Re-assemble the agent with Tool 1, Tool 2 **and** Tool 3

> **Note:** Same agent pattern as Part 3.3.2, but now the LLM has **three** tools to orchestrate: keyword, semantic, and the citation graph. Watch the model wire them together on its own.

##### 🧠 *Technical Background Notes*

In Part 3.4.1 you ran the full hybrid + graph pipeline by hand: Tool 1 → Tool 2 → `parse_ids()` union → Tool 3. In this cell, we hand all three tools to the agent and let the LLM figure out that exact sequence from a single natural-language question.

A few things to notice:

- **Fresh `OpenAIChatClient` + `as_agent(...)`**: as in Part 3.3.2, we rebuild the agent from scratch so the new tool is exposed. The `tools=[...]` list is snapshotted at construction time.
- **No explicit "first do BM25, then vector, then graph" instructions.** The model has to infer the chaining from the tool descriptions you wrote in Parts 3.4, 3.7, and 3.10: this is why descriptive `@tool(description=...)` strings and `Field(description=...)` parameter docs really matter. They're the agent's user manual.
- **Tool 3 needs ids from Tools 1 and 2.** The LLM has to take the case ids from the BM25 and vector results and pass them as `input_case_ids` to `precedent_graph_search`. This is the moment the agent stops being a fancy search box and starts being an actual reasoning loop.
- **Same Seattle flooding prompt** as Parts 3.8 and 3.11 so you can compare the agent's narrative answer against the raw rows you printed by hand.
- **"Run each tool only once"** keeps the trace readable for the lab.

##### 👀 *What you'll see in the output

When you run the cell, the `// Functions the Agent Called: //` block should contain trace lines for **all three tools**, in roughly this order:

- `keyword_case_search was called` + the BM25 args the model chose (notice: the model usually picks a legal-doctrine phrase like `"common enemy"` from the fact pattern, even though the user never said those words).
- `semantic_case_search was called` + a natural-language summary of the fact pattern, with the `advanced_filtering=ALWAYS_ON` indicator.
- `precedent_graph_search was called` + the parameters line (`inputs=N  direction=both  hops=2  top_k=10  supreme_boost=2`) and `related_count_total=X  returned_top_k=Y`.

Then under `// Agent Response: //` you'll get a rendered Markdown answer that should:

- **Cite case names** drawn from *all three* tools, not just Tools 1 and 2.
- **Lean on Washington Supreme Court cases** for the binding-precedent rule statement (those are the ones Tool 3's `supreme_boost` floats to the top).
- **Mention factually similar but doctrinally distinct cases** brought in by Tool 2 (vector) and expanded by Tool 3 (graph), not just the obvious "common enemy" hits from Tool 1.

If the agent **only** calls one or two tools, that's usually a sign the tool descriptions need to be more directive about chaining (e.g., "after calling Tool 1 and Tool 2, take the union of ids and call Tool 3"). For the lab prompt above, all three should fire.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. Confirm all three tool-call trace lines appear (`keyword_case_search`, `semantic_case_search`, `precedent_graph_search`). If `precedent_graph_search` is missing, the agent skipped graph expansion: re-run and the model usually picks it up on the second try.

1. Compare the agent's final answer to the raw `graph_out` you printed in Part 3.4.1. The agent should be naming the same top-ranked cases, but framing them as a coherent legal argument instead of a pipe-delimited table.


In [ ]:
# Step 1: Build a fresh Azure OpenAI chat client.
# Same pattern as Part 3.3.2: OpenAIChatClient + azure_endpoint routes through
# your Azure OpenAI deployment, and `model` is the *deployment name* from
# Azure AI Foundry.
client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

# Step 2: Rebuild the agent with ALL THREE retrieval tools.
# `as_agent()` snapshots the tools list at construction time, so we
# re-create the agent each time we add a new tool. The LLM now has to
# decide on its own to: (a) call keyword + semantic to gather candidate
# ids, (b) take the union, (c) pass them to precedent_graph_search as
# `input_case_ids`. The tool descriptions we wrote earlier are what teach
# it that chaining.
agent = client.as_agent(
    instructions=(
        "You are a helpful legal assistant. Respond with the case names, why each is relevant. Run each tool only once."
    ),
    tools=[
        keyword_case_search,
        semantic_case_search,
        precedent_graph_search
    ],
)

# Step 3: Same Seattle flooding scenario as Parts 3.8 and 3.11, so the
# narrative answer below is directly comparable to the raw rows you
# printed by hand a few cells ago.
user_query = (
    "I have a client in Seattle, WA whose property keeps flooding after the developer "
    "next door regraded their lot and the city redid the drainage on the road."
)

# Step 4: Run the agent. Under the hood Agent Framework will:
#   (a) send the prompt + all three tool schemas to Azure OpenAI,
#   (b) execute each tool call the model requests (printing the
#       "was called" trace lines from inside each @tool function),
#   (c) feed each tool's output back into the conversation so the next
#       tool call (and the final answer) can use it,
#   (d) loop until the model returns a natural-language answer.
print("// Functions the Agent Called: //")
result = await agent.run(user_query)

# Step 5: Render the agent's final answer as Markdown. Expect case names
# pulled from all three tools, with Washington Supreme Court precedents
# (boosted by Tool 3) doing most of the rule-statement work.
print("")
print("// Agent Response: //")
Markdown(result.text)


### Part 3.5: Tool 4 - `case_analyst_extract` (in-database entity extraction with `azure_ai`)

> **Note:** This is the **fourth** tool in our agent's pipeline. Tools 1-3 *find* the relevant cases; Tool 4 *reads* them and pulls out structured legal facts so the LLM can write a brief, not just a list.

##### 🧠 *Technical Background Notes*

**Purpose:** take the case ids that came out of `precedent_graph_search` (the `RELATED_CASE_IDS:` header line from Tool 3) and, for each one, extract a small set of structured legal entities from the opinion text: `holding`, `issues`, `statutes_cited`, and `disposition`. The output is one JSON record per case, prefixed with a stable `CASE_BRIEFS_JSON:` header so the agent (and any downstream code) can parse it deterministically.

**🌟 The `azure_ai` Postgres extension - LLM calls *from inside* the database:** the key move here is `SELECT azure_ai.extract(text, labels, model)`. That is a **SQL function** that calls an Azure OpenAI chat deployment from within the database, gets back structured JSON for the requested labels, and returns it as the query result. This tool does **not** have to ship the OpenAI key around or hand-roll a `requests.post` to the chat completions API. It is a clean separation: app code orchestrates, the database handles the model call and the data join in one round trip.

**🌟 Model registration via `model_registry.model_add(...)`:** the endpoint, deployment, API version, and subscription key were registered once in Notebook 1 against your chat deployment using something like:

```sql
SELECT model_registry.model_add(
    %s,                 -- model alias
    %s,                 -- endpoint URL
    %s,                 -- deployment name
    %s,                 -- model name
    %s,                 -- API version
    'subscription-key', -- auth type
    %s                  -- endpoint key
);
```

From then on, every `azure_ai.extract(text, labels, AZURE_OPENAI_DEPLOYMENT)` call resolves the deployment through the model registry and uses the stored endpoint + key. The huge win: you can **rotate the endpoint, key, or API version** by re-running `model_add` (or its update sibling) without changing a single line of tool code.

So in the code below you'll see:

```python
extract_sql   = "SELECT azure_ai.extract(%s::text, %s::text[], %s::text);"
extract_model = chat_deployment   # AZURE_OPENAI_DEPLOYMENT from your .env
```

The deployment name is the only credential-related value the tool needs to know - everything else (URL, key, API version) is looked up by the `azure_ai` extension via the model registry entry.

**Parallel extraction with `ThreadPoolExecutor`:** Tool 3 hands us up to 10 case ids, and each extraction is an independent LLM call. Running them sequentially would be 10x slower than necessary, so each future opens **its own** `psycopg.connect(...)` (psycopg connections are not thread-safe to share) and the pool fans them out concurrently. Results are re-sorted back into the input id order afterward so the JSON is deterministic.

**Why `max_opinion_chars = 1200`:** WA case opinions can be tens of thousands of characters, and the lab is shooting for snappy demo latency. Truncating to the first ~1200 chars captures the holding and core issues in almost every case while keeping each `extract` call cheap and fast. In a real product you would chunk and aggregate, not truncate.

**Required explicit casts (`::text`, `::text[]`):** psycopg sends parameters with the SQL type `unknown`, and PostgreSQL cannot pick the right `azure_ai.extract(...)` overload without explicit type hints. Forgetting the casts surfaces as a confusing "function does not exist" error: the function exists, but PG cannot resolve the overload.

**Tool-call trace lines** (`case_analyst_extract was called`, the `case_ids=` / `fields=` echo, and one `extracted_case=...|name` line per completed future) are what the Gradio "Tool Trace" panel keys off later in the notebook.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. This cell only **defines** the tool function, so the output is just a success check mark. We'll invoke it (against the ids from Tool 3) in the next cell.


In [ ]:
@tool(description=(
    "Extract structured legal facts from one or more case opinions using azure_ai.extract. "
    "Returns JSON per case with compact fields (holding and issues by default). "
    "Use after the graph tool to turn raw opinions into brief-ready structured records."
))
def case_analyst_extract(
    case_ids: Annotated[List[int], Field(
        description=(
            "List of case ids to analyze (typically from precedent_graph_search's RELATED_CASE_IDS)."
        )
    )]
) -> str:

    # The chat deployment name (set in your .env) is the model that
    # azure_ai.extract will call from inside the database. It is also
    # what the agent uses for its own reasoning, but here we are using
    # it specifically for structured extraction over case opinions.
    chat_deployment = AZURE_OPENAI_DEPLOYMENT

    # Trace lines for the Gradio "Tool Trace" panel later in the notebook.
    print("case_analyst_extract was called")
    print(f"  case_ids={len(case_ids)}  chat_deployment={chat_deployment}")

    # Defensive: if Tool 3 returned no related cases, return an empty
    # JSON envelope so downstream code does not blow up parsing it.
    if not case_ids:
        return "CASE_BRIEFS_JSON: []"

    # The structured fields we want azure_ai.extract to pull out of every
    # opinion. Keep this list short and stable: the agent's downstream
    # prompt (in Part 3.8 / the flagship_prompt) is written assuming
    # these exact keys exist on every extracted card.
    fields = ["holding", "issues", "statutes_cited", "disposition"]

    # Emit the entity fields the agent is extracting (parsed by the UI trace panel)
    print(f"  fields={','.join(fields)}")

    # Demo-latency cap: truncate each opinion to ~1200 chars before
    # sending it to azure_ai.extract. In production you would chunk and
    # aggregate, not truncate.
    max_opinion_chars = 1200

    # Step 1: Pull the case rows we need (id, name, date, court, opinion)
    # in a single relational query. ANY(%s) lets us pass the whole id
    # list as one parameter instead of building a comma-joined string.
    # CREATE EXTENSION IF NOT EXISTS azure_ai is idempotent and just
    # makes the extension safe to call from this connection.
    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS azure_ai;")
        cur.execute(
            """
            SELECT id, name, decision_date, court_level, opinion
            FROM public.cases
            WHERE id = ANY(%s);
            """,
            (case_ids,),
        )
        rows = cur.fetchall()

    # If somehow none of the requested ids exist in the table (e.g., a
    # bad handoff from Tool 3), short-circuit with an empty envelope.
    if not rows:
        return "CASE_BRIEFS_JSON: []"

    # Step 2: Extract in parallel - each thread opens its own connection
    # so the LLM calls run concurrently instead of sequentially.
  
    extract_sql = "SELECT azure_ai.extract(%s::text, %s::text[], %s::text);"
    extract_model = chat_deployment

    # Worker that runs inside each thread: one DB connection per thread
    # (psycopg connections are NOT thread-safe to share), one azure_ai.extract
    # call per case. Wrapping the call in try/except means a single bad
    # opinion will not nuke the whole batch.
    def _extract_one(row):
        case_id, name, decision_date, court_level, opinion = row
        short_opinion = (opinion or "")[:max_opinion_chars]
        try:
            with psycopg.connect(**DB_CONFIG, autocommit=True) as c2, c2.cursor() as cur2:
                cur2.execute(
                    extract_sql,
                    (short_opinion, fields, extract_model),
                )
                # azure_ai.extract returns JSON; psycopg already decodes
                # JSONB columns to a Python dict, so no json.loads needed.
                extracted = cur2.fetchone()[0]
        except Exception as e:
            extracted = {"error": "extraction_failed", "reason": str(e)}

        # One JSON card per case, in the exact shape the agent's downstream
        # prompt expects.
        card = {
            "id": case_id,
            "name": name,
            "decision_date": str(decision_date),
            "court_level": court_level,
            "extracted": extracted,
        }
        return card

    # Fan out: one worker per row so all extractions run concurrently.
    # as_completed() streams results back in finish order, which is also
    # the natural order to emit the per-case trace lines for the UI.
    with ThreadPoolExecutor(max_workers=len(rows)) as pool:
        futures = {pool.submit(_extract_one, r): r[0] for r in rows}
        results = []
        for fut in as_completed(futures):
            card = fut.result()
            results.append(card)
            # Emit per-case completion line for the UI trace panel
            cname = (card.get("name") or "").replace("|", "/")[:60]
            print(f"  extracted_case={card.get('id')}|{cname}", flush=True)

    # Step 3: Re-sort the results into the original input id order so the
    # JSON output is deterministic (the parallel pool returns them in
    # whatever order finished first, which is not useful for the agent).
    id_order = {cid: i for i, cid in enumerate(case_ids)}
    results.sort(key=lambda c: id_order.get(c["id"], 1_000_000))

    # Stable header + indented JSON body. The "CASE_BRIEFS_JSON:" prefix
    # is the parseable handoff marker that the agent (and the Gradio UI)
    # looks for when picking the extracted briefs out of this tool's output.
    return "CASE_BRIEFS_JSON: " + json.dumps(results, indent=2, default=str)


### Part 3.5.1: Re-assemble the agent with Tool 1, Tool 2, Tool 3 **and** Tool 4

> **Note:** Same agent pattern as Parts 3.9 and 3.12, but now the LLM has **four** tools. Tool 4 is the first one that doesn't *find* cases - it *reads* the ones already found and pulls structured facts out of them.

##### 🧠 *Technical Background Notes*

In Part 3.4.2 the agent could find and rank cases, but its final answer was still based on whatever short opinion snippet got into Tool 3's pipe-delimited output. Now we add `case_analyst_extract`, and the LLM gains an in-database **read + extract** step: it can take the `RELATED_CASE_IDS:` line from `precedent_graph_search` and ask `azure_ai.extract` to pull the `holding`, `issues`, `statutes_cited`, and `disposition` out of each opinion before composing the brief.

A few things to notice:

- **Fresh `OpenAIChatClient` + `as_agent(...)`**: as before, we rebuild the agent so the new tool is exposed. The `tools=[...]` list is snapshotted at construction time.
- **Expected chain**: `keyword_case_search` -> `semantic_case_search` -> union of ids -> `precedent_graph_search` -> parse `RELATED_CASE_IDS:` -> `case_analyst_extract`. The LLM has to infer that whole sequence from the tool descriptions alone.
- **Tool 4 needs ids from Tool 3.** Specifically, it expects the ids that appeared on Tool 3's `RELATED_CASE_IDS:` line. If the model passes the Tool 1 + Tool 2 union instead, it's skipping the graph expansion - watch for that in the trace.
- **Same Seattle flooding prompt** as Parts 3.8, 3.11, and 3.12 so the narrative answer is directly comparable.
- **"Run each tool only once"** keeps the trace readable for the lab.

##### 👀 *What you'll see in the output*

When you run the cell, the `// Functions the Agent Called: //` block should contain trace lines for **all four tools**, in roughly this order:

- `keyword_case_search was called` + the BM25 args the model chose.
- `semantic_case_search was called` + the natural-language fact pattern, with `advanced_filtering=ALWAYS_ON`.
- `precedent_graph_search was called` + the parameters line (`inputs=N  direction=both  hops=2  top_k=10  supreme_boost=2`) and `related_count_total=X  returned_top_k=Y`.
- `case_analyst_extract was called` + `case_ids=N  chat_deployment=...`, the `fields=holding,issues,statutes_cited,disposition` echo, and one `extracted_case=<id>|<name>` line per case as the parallel extractions complete (order will be whatever finished first).

Then under `// Agent Response: //` you'll get a rendered Markdown answer that should:

- **Quote or paraphrase the actual `holding`** for the top cases, not just the case name and a one-line snippet.
- **Mention the `issues`** the court was deciding, not just the doctrine label.
- **Cite statutes** when Tool 4's `statutes_cited` field surfaced any, e.g. WA Code sections.
- **Read like a junior associate's brief**: rule statement, supporting precedents, factual analogues. That qualitative jump versus Part 3.4.2 is the whole reason Tool 4 exists.

If `case_analyst_extract` is missing from the trace, the agent did retrieval but skipped extraction: re-run, and the model usually picks it up on the second pass. If individual `extracted_case=...` lines are missing, the `try/except` inside the worker absorbed an extraction failure for that case - the JSON envelope will contain `{"error": "extraction_failed", ...}` for it, and the agent will quietly work around it.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. Confirm all four tool-call trace lines appear, ending with at least a few `extracted_case=...` lines from the parallel pool.

1. Compare this agent's answer to the Part 3.4.2 answer. The case names should overlap heavily, but this one should sound like an actual legal memo - holdings, issues, statutes - instead of "here are some relevant cases".


In [ ]:
# Step 1: Build a fresh Azure OpenAI chat client.
# Same pattern as Parts 3.9 and 3.12: OpenAIChatClient + azure_endpoint
# routes through your Azure OpenAI deployment. `model` is the *deployment
# name* from Azure AI Foundry, not the base model id.
client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

# Step 2: Rebuild the agent with ALL FOUR tools.
# `as_agent()` snapshots the tools list at construction time, so we
# re-create the agent each time we add a new tool.
#
# The LLM now has to chain: keyword + semantic to gather candidate ids ->
# union them -> precedent_graph_search to expand via the citation graph ->
# parse the RELATED_CASE_IDS line out of Tool 3's output -> case_analyst_extract
# to pull structured holdings/issues/statutes/disposition out of each
# opinion. The tool descriptions you wrote earlier are what teach the
# model that whole sequence.
agent = client.as_agent(
    instructions=(
        "You are a helpful legal assistant. Respond with holdings, statutes, issues, and dispositions. Cite the case names, why each is relevant. Run every tool only once."
    ),
    tools=[
        keyword_case_search,
        semantic_case_search,
        precedent_graph_search,
        case_analyst_extract,
    ],
)

# Step 3: Same Seattle flooding scenario as Parts 3.8, 3.11, and 3.12,
# so the narrative answer below is directly comparable to the earlier
# agent runs. The interesting question is: does the answer get
# qualitatively better now that Tool 4 is in the mix?
user_query = (
    "I have a client in Seattle, WA whose property keeps flooding after the developer "
    "next door regraded their lot and the city redid the drainage on the road."
)

# Step 4: Run the agent. Under the hood Agent Framework will:
#   (a) send the prompt + all four tool schemas to Azure OpenAI,
#   (b) execute each tool call the model requests (printing the
#       "was called" trace lines from inside each @tool function;
#       Tool 4 also prints one "extracted_case=..." line per case as
#       the ThreadPoolExecutor workers finish),
#   (c) feed each tool's output back into the conversation so the next
#       tool call (and the final answer) can use it,
#   (d) loop until the model returns a natural-language answer.
print("// Functions the Agent Called: //")
result = await agent.run(user_query)

# Step 5: Render the agent's final answer as Markdown. Expect a memo
# that quotes actual holdings, lists the issues each court decided, and
# cites statutes when Tool 4's `statutes_cited` field surfaced any.
# That qualitative jump versus Part 3.4.2 is what Tool 4 buys you.
print("")
print("// Agent Response: //")
Markdown(result.text)


### Part 3.6: Tool 5 - `get_weather_evidence` (external evidence from the Open-Meteo API)

> **Note:** This is the **fifth and final** tool in our agent's pipeline. The first four tools all talk to HorizonDB. This one talks to the **outside world**: a public HTTP API for historical weather data.

##### 🧠 *Technical Background Notes*

**Purpose:** for legal questions that turn on weather (flooding, storm damage, drainage disputes, "did it actually rain that day?"), let the agent pull historical precipitation records from the **Open-Meteo Archive API** for a specific lat/lon and date range. The output is a JSON summary (total mm of precipitation, days with rain, daily records) prefixed with a `WEATHER_EVIDENCE_JSON:` header so the agent can parse it deterministically alongside the `CASE_BRIEFS_JSON:` payload from Tool 4.

**🌟 Why mix an external API into a database-centric agent?** Agents in the real world rarely live in just one data source. The first four tools demonstrate everything HorizonDB can do on its own (FTS, vector, graph, in-database LLM extraction). Tool 5 demonstrates the equally important pattern of **reaching out** when the relevant evidence is somewhere else entirely - here, a free public API. Same `@tool` decorator, same `Annotated[..., Field(...)]` schema pattern: the LLM does not know or care that this tool happens to use `requests` instead of `psycopg`.

**Why the description says "OPTIONAL":** Tools 1-4 are **always** relevant to a legal question. Tool 5 is only relevant when the case turns on weather facts. The `@tool` description deliberately spells out the trigger words (`rain, weather, water, storms, flooding, drainage, precipitation`) so the LLM knows when to invoke it and when to skip it. This is the right way to encode tool selection logic for an agent: in the natural-language description, not in brittle keyword routing on the host side. For our Seattle flooding fact pattern, the model **should** decide to call this tool. For a contract dispute prompt, it should not.

**Required vs optional arguments:** `latitude`, `longitude`, and `start_date` are required, so the LLM is forced to pin down a location and a date before calling. `end_date` is optional and defaults to `start_date`, which means the agent can ask for a single day without having to think about a range. This is the same pattern used in Tool 2 where required filter args (`court_level`, `min_year`) guarantee the model supplies the right shape of input.

**Open-Meteo Archive API:** free, no API key required, queries by lat/lon and date range, returns daily aggregates. We ask only for `daily: precipitation_sum` to keep the payload small. The 10-second `timeout` and `raise_for_status()` plus `try/except` keep a flaky network from breaking the entire agent run: on failure the tool returns a `WEATHER_EVIDENCE_ERROR: ...` string instead of raising, so the agent can carry on with what it has from Tools 1-4.

**Payload trimming - `records if len(records) <= 14 else records[:7] + records[-7:]`:** if the date range is short (≤14 days) we return every daily record; otherwise we return the first 7 and the last 7 days plus the aggregate totals. Keeps the LLM context window happy on longer date ranges (e.g., "the whole rainy season") without losing the summary or the bookend dates.

**Tool-call trace lines** (`get_weather_evidence was called` and the `lat= lon= start= end=` echo) are what the Gradio "Tool Trace" panel keys off later in the notebook.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. This cell only **defines** the tool function, so the output is just a success check mark. We'll plug it into the full five-tool agent in the next section.


In [ ]:
# Note the word "OPTIONAL" in the description plus the explicit list of
# trigger words (rain, weather, water, storms, flooding, drainage,
# precipitation). That sentence is the agent's user manual for *when*
# to call this tool. The LLM reads it and decides whether the user's
# question warrants pulling weather evidence at all.
@tool(description=(
    "Retrieve historical precipitation data for a location and date range to support "
    "weather-related legal claims (rain, flooding, storm damage, drainage disputes). "
    "Uses the Open-Meteo Archive API. This tool is OPTIONAL - only call it when the "
    "legal question involves rain, weather, water, storms, flooding, drainage, or precipitation."
))
def get_weather_evidence(
    # latitude / longitude / start_date are REQUIRED so the LLM has to
    # commit to a specific location and date before calling. end_date is
    # optional and defaults to start_date, which lets the model ask for
    # a single day without having to think about a range.
    latitude: Annotated[float, Field(
        description="Latitude of the location (WGS84). Example: 47.6062 for Seattle."
    )],
    longitude: Annotated[float, Field(
        description="Longitude of the location (WGS84). Example: -122.3321 for Seattle."
    )],
    start_date: Annotated[str, Field(
        description="Start date in YYYY-MM-DD format."
    )],
    end_date: Annotated[Optional[str], Field(
        default=None,
        description="End date in YYYY-MM-DD format. Defaults to start_date (single day)."
    )] = None,
) -> str:
    # Trace lines for the Gradio "Tool Trace" panel later in the notebook.
    print("get_weather_evidence was called")
    print(f"  lat={latitude}  lon={longitude}  start={start_date}  end={end_date or start_date}")

    # Single-day query convenience: if the model only supplied a start
    # date, treat the range as that one day.
    if end_date is None:
        end_date = start_date

    # Step 1: Build the Open-Meteo Archive API request.
    # Free, no API key required. We only ask for daily precipitation_sum
    # to keep the payload small and the LLM context window happy.
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "precipitation_sum",
        "timezone": "UTC",
    }

    # Step 2: Fire the request with a 10-second timeout so a slow network
    # cannot stall the entire agent run. Any failure (timeout, HTTP error,
    # bad JSON) is caught and surfaced as a WEATHER_EVIDENCE_ERROR string
    # rather than a Python exception - the agent can then carry on with
    # what it already has from Tools 1-4.
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        return f"WEATHER_EVIDENCE_ERROR: Failed to retrieve weather data: {e}"

    # Step 3: Parse the parallel arrays Open-Meteo returns (dates[] and
    # precipitation_sum[]). Defensive .get(...) chains so a malformed
    # response gracefully degrades to "no data" instead of crashing.
    dates = data.get("daily", {}).get("time", [])
    precip = data.get("daily", {}).get("precipitation_sum", [])

    if not dates:
        return "WEATHER_EVIDENCE: No data returned for the given parameters."

    # Step 4: Aggregate. None values are treated as zero so a single
    # missing day does not poison the total. We also count "days with
    # any rain" because it is a useful summary statistic for a brief.
    records = []
    total_mm = 0.0
    for d, p in zip(dates, precip):
        rain = p if p is not None else 0.0
        total_mm += rain
        records.append({"date": d, "precipitation_mm": rain})

    # Step 5: Build the summary payload. For short ranges we include
    # every daily record. For longer ranges (>14 days) we keep only the
    # first 7 and last 7 days plus the aggregate totals, so the JSON
    # stays small even for "the whole rainy season" style queries.
    summary = {
        "location": {"latitude": latitude, "longitude": longitude},
        "period": {"start": start_date, "end": end_date},
        "total_precipitation_mm": round(total_mm, 2),
        "days_with_rain": sum(1 for r in records if r["precipitation_mm"] > 0),
        "total_days": len(records),
        "daily_records": records if len(records) <= 14 else records[:7] + records[-7:],
    }

    # Stable header prefix so the agent (and the Gradio UI parser) can
    # pick this payload out of the conversation later, exactly like the
    # CASE_BRIEFS_JSON: header used by Tool 4.
    return "WEATHER_EVIDENCE_JSON: " + json.dumps(summary, ensure_ascii=False)


#### Part 3.6.1: Smoke-test Tool 5 directly

> **Note:** Same pattern as the other smoke tests (Parts 3.2.1, 3.3.1, 3.4.1): call `get_weather_evidence` by hand with known-good arguments before handing it to the agent in Part 3.7. If the raw call works, any failure in the next cell is the agent's tool-selection logic, not the tool itself.

##### 🧠 *Technical Background Notes*

**Purpose:** confirm Tool 5 talks to the Open-Meteo API correctly and returns the expected `WEATHER_EVIDENCE_JSON:` envelope **before** the agent is allowed to choose it. Smoke-testing tools by hand makes the next cell's agent trace much easier to debug: if the JSON looks right here, you know the tool is fine.

**🌟 Why we hard-code the arguments:** the agent will eventually derive `latitude`, `longitude`, `start_date`, and `end_date` from a natural-language prompt. At this stage we are **not** testing that derivation, we are testing the HTTP call and the parsing logic inside `get_weather_evidence`. So we pass coordinates (Seattle, 47.6062 / -122.3321) and a 10-day window in January 2023 directly.

**🌟 Why parse the JSON envelope:** Tool 5 returns a single string with a stable `WEATHER_EVIDENCE_JSON: {...}` prefix (defined in Part 3.6). The prefix is the parseable handoff marker the agent uses to extract the structured payload from a tool message. In this smoke test we mimic that exact behavior: split on the prefix, `json.loads` the rest, and pretty-print with `indent=2` so a student can eyeball the daily `precip_mm`, `tmax_c`, and `tmin_c` rows.

**🌟 What the JSON tells you:** Open-Meteo's archive endpoint returns one row per day in the requested window. The agent will later use the `precip_mm` series to corroborate or contradict a client's "it rained for ten days straight" claim, which is exactly the kind of objective external evidence a legal brief needs.

##### 👀 *What you'll see in the output*

A `// Tool 5: Weather Evidence ... //` banner, then a `=== Weather Evidence (pretty) ===` block with a JSON object containing a `location` and a `daily` array (or list of day records) for Seattle, Jan 10-20 2023. Daily entries should show realistic Pacific Northwest winter values: temperatures around 0-10 C and precipitation in the 0-20 mm range on most days. If the response starts with anything other than `WEATHER_EVIDENCE_JSON:` (an error string), the `else` branch will print it raw so you can see what went wrong.

##### 📝 *Tasks*

1. Run the cell using the "▶" icon next to the cell.
1. Scan the daily rows. Pick out which days had the highest `precip_mm`: those are the days that would matter to a flooding case.
1. Confirm the JSON structure looks like a real `daily` time series (one entry per day, three numeric fields each). If you see an HTTP error message instead, check that the lab VM has outbound network access to `archive-api.open-meteo.com`.


In [ ]:
# Banner so this block is easy to spot when scrolling through the
# notebook output. The // ... // delimiters match the style used in
# the earlier smoke-test cells.
print("// Tool 5: Weather Evidence (optional - weather-related query) //")

# Step 1: Call Tool 5 directly with hard-coded arguments.
# Coordinates are downtown Seattle (47.6062 N, -122.3321 W) and the
# date window is 10 days in January 2023. We are bypassing the agent
# on purpose: the goal is to verify the HTTP call to Open-Meteo and
# the JSON parsing inside get_weather_evidence, NOT the agent's
# ability to extract these arguments from a natural-language prompt.
weather_out = get_weather_evidence(
    latitude=47.6062,
    longitude=-122.3321,
    start_date="2023-01-10",
    end_date="2023-01-20",
)

# Step 2: Tool 5 returns a single string. On success it has the stable
# "WEATHER_EVIDENCE_JSON: {...}" envelope (the parseable handoff the
# agent and the Gradio UI key off later). On failure it returns a
# plain error string with no prefix.
#
# Here we split on that exact prefix, json.loads the payload, and
# pretty-print so a student can eyeball the daily precip / temp rows.
# This mirrors what the agent will do internally when it consumes the
# tool response in Part 3.7.
if weather_out.startswith("WEATHER_EVIDENCE_JSON:"):
    wdata = json.loads(weather_out.replace("WEATHER_EVIDENCE_JSON: ", ""))
    print("\n=== Weather Evidence (pretty) ===")
    print(json.dumps(wdata, indent=2, ensure_ascii=False))
else:
    # Error path: print the raw string so the failure mode is visible.
    print(weather_out)


#### Part 3.6.2: Re-assemble the agent with **all five tools** (the flagship run)

> **Note:** This is it - the full agentic legal-research pipeline. One natural-language question, five tools, a structured legal brief at the end.

##### 🧠 *Technical Background Notes*

In Part 3.5.1 the agent could find, expand, and extract structured facts from cases. In this cell we add `get_weather_evidence` and beef up the system prompt so the LLM produces a real legal analysis with holdings, statutes, issues, and dispositions - not just a list of relevant cases.

A few things to notice:

- **Fresh `OpenAIChatClient` + `as_agent(...)`**: same rebuild-from-scratch pattern as Parts 3.9 / 3.12 / 3.14. The `tools=[...]` list is snapshotted at construction time.
- **Beefier system prompt**: this one is more directive than the previous runs. It tells the agent (a) what shape the answer should take ("holdings, statutes, issues, and dispositions"), (b) when to invoke the optional weather tool ("If the question involves weather, flooding, storms, precipitation, drainage, or water damage"), and (c) keeps the lab-friendly "Run every tool only once" guardrail so the trace stays readable.
- **Conditional tool**: `get_weather_evidence` was declared OPTIONAL in its `@tool` description (Part 3.6). The prompt above reinforces *when* to use it. Because the new user prompt explicitly says "rain storm" + "February of 2024", the model **should** decide to call it. On a prompt with no weather signal, it would skip it.
- **Richer user prompt**: this one also asks for a "structured legal analysis and draft brief", which is the signal the model needs to actually compose a memo instead of summarizing cases.
- **Same Seattle flooding fact pattern** as the earlier runs, but with a concrete date (February 2024) so the weather tool has something to query.

##### 👀 *What you'll see in the output*

When you run the cell, the `// Functions the Agent Called: //` block should contain trace lines for **all five tools**, in roughly this order:

- `keyword_case_search was called` + the BM25 args the model picked.
- `semantic_case_search was called` + the natural-language fact pattern, with `advanced_filtering=ALWAYS_ON`.
- `precedent_graph_search was called` + the parameters line and the `related_count_total=X  returned_top_k=Y` summary.
- `case_analyst_extract was called` + the `fields=` echo and one `extracted_case=<id>|<name>` line per case as the parallel extractions finish.
- `get_weather_evidence was called` + `lat=47.6062  lon=-122.3321  start=2024-02-...  end=2024-02-...`. The lat/lon should be Seattle; the dates should be in February 2024.

Then under `// Agent Response: //` you'll get a rendered Markdown answer that should:

- **Read like a draft brief**, not a list. Expect headings or numbered sections.
- **Quote real holdings** pulled by Tool 4, with case names and dates.
- **Cite statutes** from Tool 4's `statutes_cited` field when any surfaced.
- **Reference actual rainfall numbers** from Tool 5 ("approximately N mm of precipitation over M days") to back the factual side of the flooding claim.
- **Tie the weather evidence into the legal analysis**: the whole point of mixing in Tool 5 is to let the agent say "the precipitation record supports the factual basis for the surface-water claim because...".

If `get_weather_evidence` is missing from the trace, the agent did the legal research but skipped the weather facts: the prompt language is what triggers it, so re-running usually gets the model to pick it up.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. Confirm all five tool-call trace lines appear, with `get_weather_evidence` showing Seattle coordinates and February 2024 dates.

1. Compare this answer to the Part 3.5.1 answer (four tools). The case research should be similar, but this one should weave the rainfall data into the legal analysis. That cross-domain synthesis - case law + structured extracts + external weather evidence, in a single agent run - is the headline capability of the lab.


In [ ]:
# Step 1: Build a fresh Azure OpenAI chat client.
# Same pattern as every other re-assembly cell (Parts 3.9 / 3.12 / 3.14):
# OpenAIChatClient + azure_endpoint routes through your Azure OpenAI
# deployment, `model` is the deployment name from Azure AI Foundry.
client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

# Step 2: Rebuild the agent with ALL FIVE tools and a beefier system prompt.
#
# The instructions string is the agent's "house rules":
#   - "Respond with holdings, statutes, issues, and dispositions" tells
#     the LLM what shape the final answer should take (it lines up
#     exactly with the fields case_analyst_extract pulls in Tool 4).
#   - The "If the question involves weather..." sentence reinforces the
#     OPTIONAL trigger that get_weather_evidence already advertises in
#     its @tool description, making it more likely the model invokes
#     Tool 5 on the prompt below.
#   - "Run every tool only once" keeps the tool trace readable for the lab.
agent = client.as_agent(
    instructions=(
        "You are a helpful legal assistant. Respond with holdings, statutes, issues, and dispositions."
        "If the question involves weather, flooding, storms, precipitation, drainage, or water damage, then call the get_weather_evidence tool."
        "Cite the case names, why each is relevant. Run every tool only once."
    ),
    tools=[
        keyword_case_search,
        semantic_case_search,
        precedent_graph_search,
        case_analyst_extract,
        get_weather_evidence,
    ],
)

# Step 3: The flagship user prompt.
# Adds two things versus the earlier runs:
#   - A concrete date ("February of 2024") so get_weather_evidence has
#     a real date range to query the Open-Meteo API for.
#   - An explicit ask for "a structured legal analysis and draft brief"
#     so the LLM composes a memo instead of a bullet list of cases.
user_query = (
    "I have a client in Seattle, Washington, whose property floods after a neighboring developer "
    "regraded their lot and in February of 2024 there was a rain storm, and the city modified road drainage."
    "Analyze potential legal liability and produce a structured legal analysis and draft brief."
)

# Step 4: Run the agent. Under the hood Agent Framework will:
#   (a) send the prompt + all FIVE tool schemas to Azure OpenAI,
#   (b) execute each tool call the model requests (printing the
#       "was called" trace lines, plus Tool 4's per-case
#       "extracted_case=..." lines and Tool 5's lat/lon/date echo),
#   (c) feed each tool's output back into the conversation,
#   (d) loop until the model returns a natural-language answer.
print("// Functions the Agent Called: //")
result = await agent.run(user_query)

# Step 5: Render the final answer as Markdown. Expect a draft legal
# brief that quotes real holdings (from Tool 4), cites statutes when
# they surfaced, and weaves the February 2024 rainfall totals (from
# Tool 5) into the factual side of the analysis. That cross-domain
# synthesis is the headline capability of this notebook.
print("")
print("// Agent Response: //")
Markdown(result.text)


### Part 3.7: Add long-term memory to the agent with Mem0 (stored in HorizonDB)

> **Note:** Every agent run up to this point has been **stateless**. The model has no idea what the client said five minutes ago, let alone last week. Mem0 fixes that, and (the headline feature) it stores those memories right back in HorizonDB.

##### 🧠 *Technical Background Notes*

**Purpose:** give the agent a **persistent, per-client memory** so it can remember the client's name, where the property is, what storm they mentioned, and which legal theory you ran with - across turns, across cells, across days. Mem0 is an open-source memory layer that, after every conversation turn, asks an LLM to extract durable facts ("client lives in Queen Anne, Seattle"; "storm was in November 2023") and writes them as embeddings to a vector store. Before the next turn it embeds the new user message and pulls back the top-k relevant memories.

**🌟 Mem0 stores everything inside HorizonDB via `pgvector`:** Mem0 supports a pgvector backend natively, so the memory collection (`counsel_agent_memory`) lives in **the same Postgres** that already holds the case rows, the AGE citation graph, and the opinion embeddings. No second service to provision, no extra credentials, no cross-system joins. This is the same "everything in one Postgres" theme the rest of the notebook has been hammering on.

**The three pieces of the Mem0 config:**

- **`vector_store: pgvector`**: re-uses the same `AZURE_PG_*` env vars `DB_CONFIG` uses. `embedding_model_dims=1536` matches Azure OpenAI's `text-embedding-3-small` output size, and `collection_name="counsel_agent_memory"` is the table Mem0 creates and writes to.
- **`llm: azure_openai`**: the chat model Mem0 uses to **extract** memories from each `{role: ..., content: ...}` pair. It is the same deployment the agent itself uses, just wearing a different hat.
- **`embedder: azure_openai`**: the embedding model Mem0 uses to **index and search** the stored memories. Same deployment as the case opinions, so search-quality is consistent across the system.

**The two-step turn lifecycle (`run_turn`):**

1. `memory.search(query=user_query, user_id=USER_ID, limit=5)` returns the top-5 most relevant memories for this user. We render them as a bullet list and inject them into the agent's system prompt under a "Long-term memory about this client" heading. The LLM then has those facts in context for the whole turn.
2. After the agent answers, `memory.add([{user...}, {assistant...}], user_id=...)` hands both sides of the turn back to Mem0. Mem0's LLM extracts whatever new durable facts it can ("the city repaved the road in 2024"; "client wants a brief, not a memo") and appends them to the per-user store.

**Per-user namespacing**: every Mem0 call is scoped by `user_id=USER_ID`, so memories about one client never leak into another. Changing `USER_ID` to a new value gives you a fresh slate without dropping the underlying table.

**Why rebuild the agent every turn** (`build_counsel_agent(memory_block)`): the memory block is baked into the system prompt at agent construction time. Re-creating the agent each turn is the cleanest way to push freshly-retrieved memories into that prompt. Cheap to do, and it keeps the wiring obvious.

##### 👀 *What you'll see in the output*

When you run the cell, each turn prints three labeled blocks:

- **`=== USER ===`**: the prompt you sent.
- **`--- Memories retrieved for this turn ---`**: the bullet list Mem0 surfaced before calling the agent. On Turn 1 this will be `(no prior memories for this user)`. On later turns expect to see facts like `- Client owns a property in the Queen Anne neighborhood of Seattle, Washington`.
- **`--- New memories extracted from this turn ---`**: the new durable facts Mem0 wrote back to HorizonDB after the agent's response. These are what later turns will be able to recall.
- **`=== AGENT RESPONSE ===`**: the rendered Markdown answer. On Turn 2 onwards you should notice the agent referring to the client by location ("your client in Queen Anne") and *not* asking intake questions it already has the answers to.

##### 📝 *Tasks*

The cell deliberately defines **four `user_query` blocks**, each one re-assigning the same variable. Whichever one is **last** is the one that runs. Step through them one at a time:

1. **Turn 1 - intake**: leave only the first `user_query` uncommented (the others can stay; just make sure Turn 1 is the last assignment). Run the cell. You should see `(no prior memories...)` retrieved and one or two new intake facts extracted.

1. **Turn 2 - describe the problem**: move the comment so Turn 2 is the last assignment. Re-run. Confirm Turn 1's intake facts now appear under "Memories retrieved", and that the agent's response references them without re-asking.

1. **Turn 3 - follow-up that depends on Turn 2**: same drill. The agent should reference the legal theory it landed on in Turn 2 (e.g., "common enemy doctrine") without you re-stating it.

1. **Turn 4 - long-term recall test**: ask the agent to remind you of the neighborhood and storm month. It should answer directly from memory without calling any retrieval tools.


In [ ]:
# Stable user id - Mem0 namespaces memories per user, so memories about
# this client never leak into another client's session. Change this to
# a new value to start fresh without dropping the underlying table.
USER_ID = "client-001"

# Mem0 configuration - three pieces:
#   - vector_store: pgvector in HorizonDB (re-using AZURE_PG_* env vars).
#                   This is the headline integration: memories live in
#                   the SAME Postgres as the case rows, AGE graph, and
#                   opinion embeddings.
#   - llm:          Azure OpenAI chat deployment Mem0 uses to EXTRACT
#                   durable facts from each turn (same deployment the
#                   agent itself uses, just wearing a different hat).
#   - embedder:     Azure OpenAI embedding deployment Mem0 uses to
#                   INDEX and SEARCH the stored memories. 1536 dims
#                   matches text-embedding-3-small.
mem0_config = {
    "vector_store": {
        "provider": "pgvector",
        "config": {
            "host":     os.environ["AZURE_PG_HOST"],
            "port":     int(os.environ["AZURE_PG_PORT"]),
            "user":     os.environ["AZURE_PG_USER"],
            "password": os.environ["AZURE_PG_PASSWORD"],
            "dbname":   os.environ["AZURE_PG_NAME"],
            "collection_name":      "counsel_agent_memory",
            "embedding_model_dims": 1536,
        },
    },
    "llm": {
        "provider": "azure_openai",
        "config": {
            "model": os.environ["AZURE_OPENAI_DEPLOYMENT"],
            "azure_kwargs": {
                "azure_endpoint":   os.environ["AZURE_OPENAI_ENDPOINT"],
                "api_key":          os.environ["AZURE_OPENAI_KEY"],
                "api_version":      os.environ["AZURE_API_VERSION"],
                "azure_deployment": os.environ["AZURE_OPENAI_DEPLOYMENT"],
            },
        },
    },
    "embedder": {
        "provider": "azure_openai",
        "config": {
            "model": os.environ["AZURE_EMBED_DEPLOYMENT"],
            "azure_kwargs": {
                "azure_endpoint":   os.environ["AZURE_OPENAI_ENDPOINT"],
                "api_key":          os.environ["AZURE_OPENAI_KEY"],
                "api_version":      os.environ["AZURE_API_VERSION"],
                "azure_deployment": os.environ["AZURE_EMBED_DEPLOYMENT"],
            },
        },
    },
}

# Mem0 will create the `counsel_agent_memory` collection in HorizonDB
# on first use if it does not already exist.
memory = Memory.from_config(mem0_config)


def fetch_memory_context(query: str) -> str:
    """Pull the top-k memories most relevant to the upcoming user turn.

    Mem0 embeds `query` with our Azure embedding deployment, runs a
    pgvector similarity search against the `counsel_agent_memory`
    collection scoped to `user_id`, and returns the top-5 hits.
    """
    hits = memory.search(query=query, user_id=USER_ID, limit=5).get("results", [])
    if not hits:
        return "(no prior memories for this user)"
    return "\n".join(f"- {h['memory']}" for h in hits)


def build_counsel_agent(memory_block: str):
    """Re-build the agent each turn so freshly retrieved memories appear in the system prompt.

    The memory block is baked into the system prompt at agent
    construction time, so the cleanest way to expose new memories to
    the LLM is to rebuild the agent. Cheap to do; keeps the wiring
    obvious.
    """
    client = OpenAIChatClient(
        model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_KEY"],
    )
    return client.as_agent(
        instructions=(
            "You are Counsel, an AI legal associate helping analyze Washington State legal cases.\n\n"
            # The "## Long-term memory..." heading + bullet list is the
            # actual mechanism by which Mem0's recall reaches the LLM:
            # it's just text in the system prompt.
            "## Long-term memory about this client (from prior conversations)\n"
            f"{memory_block}\n\n"
            "Use the memory above to personalize answers - refer to the client by name, skip "
            "intake questions you already know the answer to, and build on facts established "
            "in earlier turns.\n\n"
            "When a question requires legal research, use your tools (keyword_case_search, "
            "semantic_case_search, precedent_graph_search, case_analyst_extract, "
            "get_weather_evidence) in that order, each at most once, then synthesize an legal analysis."
        ),
        tools=[
            keyword_case_search,
            semantic_case_search,
            precedent_graph_search,
            case_analyst_extract,
            get_weather_evidence,
        ],
    )


async def run_turn(user_query: str):
    """One conversation turn: retrieve -> run agent -> persist new memories."""
    print(f"\n=== USER ===\n{user_query}\n")

    # Step 1: RETRIEVE - pull the most relevant prior memories for this
    # user, embed-search style, BEFORE the agent runs.
    memory_block = fetch_memory_context(user_query)
    print("--- Memories retrieved for this turn ---")
    print(memory_block)

    # Step 2: RUN - rebuild the agent with the freshly retrieved memory
    # block injected into the system prompt, then run the turn.
    agent = build_counsel_agent(memory_block)
    result = await agent.run(user_query)

    # Step 3: PERSIST - hand both sides of the turn (user prompt +
    # agent response) back to Mem0. Mem0's LLM reads the pair, extracts
    # any new durable facts, and writes them to the pgvector store
    # scoped to USER_ID so the NEXT turn can recall them.
    added = memory.add(
        [
            {"role": "user",      "content": user_query},
            {"role": "assistant", "content": result.text},
        ],
        user_id=USER_ID,
    )
    new_facts = [m["memory"] for m in added.get("results", [])]
    print("\n--- New memories extracted from this turn ---")
    for f in new_facts or ["(none)"]:
        print(f"+ {f}")

    print("\n=== AGENT RESPONSE ===")
    return Markdown(result.text)


# ─────────────────────────────────────────────────────────────────────
# Conversation simulation - WHICHEVER user_query block is LAST in this
# cell wins (Python just re-assigns the same variable). To step through
# the turns, move the comment markers so the turn you want to run is
# the last assignment, then re-execute the cell.
# ─────────────────────────────────────────────────────────────────────

# ── TURN 1: initial intake (run this first) ──────────────────────────

user_query = (
    "Hi, I am an attorney and my client owns a property in the 'Queen Anne neighborhood' of Seattle, Washington."
    "Please remember those details for later. No legal question yet."
)

# ── TURN 2: describe the problem WITHOUT repeating any context ───────
# Notice this prompt never says "Queen Anne" or "Seattle" - the agent
# should pick those up from the memories retrieved at the start of the turn.

user_query = (
     "After a heavy storm last November my client's yard floods every time it rains. "
     "A neighbor of my client regraded their lot last summer and the city repaved the road. "
     "Does my client have a claim?"
)

#── TURN 3: follow-up that depends on TURN 2's analysis ──────────────
# "Given what you found" only makes sense if the agent remembers the
# legal theory it landed on in Turn 2 - that's the memory at work.

#user_query = (
#     "Given what you found, who is the most promising defendant and what "
#     "evidence should I start collecting?"
#)

# ── TURN 4: verify long-term recall ──────────────────────────────────
# Pure recall test - the agent should answer from memory alone, no
# retrieval tools needed.

#user_query = (
#     "Remind me - what neighborhood is my client's property in, and what was the month of the storm we discussed?"
#)

await run_turn(user_query)


### Part 3.8: Wrap the agent in a Gradio web UI (all 5 tools + Mem0 memory)

> **Note:** Final cell of the notebook. Everything you've built so far - five tools, hybrid retrieval, graph expansion, in-database extraction, external weather evidence, and Mem0 long-term memory - is now wired up behind a chat UI that you can actually click around in.
>
> **Once the cell finishes starting up, open the app here: [http://localhost:7860](http://localhost:7860)**
> You **must run the cell first** - the link will not work until Gradio prints `Running on local URL: http://127.0.0.1:7860` in the cell output.

##### 🧠 *Technical Background Notes*

**Purpose:** put a real chat interface in front of the Counsel agent so the lab feels like a product instead of a notebook. The UI shows three things side by side: the conversation, a live **Tool Trace** panel that surfaces each `print("... was called")` line from your tools as the agent fires them, and the **Long-term Memory** panel backed by Mem0 + pgvector.

**🌟 Why Gradio?** `gradio` lets us stand up a multi-pane web app from inside a notebook cell with no separate web server, no template files, no JavaScript. `gr.Blocks(...)` gives us the layout, `demo.launch(server_port=7860, inline=False)` starts a local web server on port 7860 and returns control to the notebook. The app keeps running until you either re-execute this cell or shut the kernel down.

**How the pieces hook up:**

- **Agent**: a fresh `OpenAIChatClient + as_agent(...)` with **all five tools** registered (`keyword_case_search`, `semantic_case_search`, `precedent_graph_search`, `case_analyst_extract`, `get_weather_evidence`). Same pattern as Part 3.7, just lifted into UI-friendly helper functions.
- **Mem0**: re-uses the **same `memory` client** initialized in Part 3.8 (pgvector in HorizonDB), but runs under its own `_UI_USER_ID` so memories from the notebook walkthrough do not leak into the UI session. Every time you re-execute this cell, that user's memories are wiped (`memory.delete_all(user_id=_UI_USER_ID)`) so the launched app always starts from a clean slate.
- **Tool Trace panel**: parses the standard `print("<tool_name> was called")` lines (plus the per-tool argument echoes you wrote earlier) into a live, structured display. This is why every tool in Parts 3.4-3.15 prints that exact pattern: the UI keys off it.
- **Long-term Memory panel**: calls `memory.search(...)` before each turn and `memory.add(...)` after each turn, just like `run_turn` in Part 3.8, and renders the retrieved/extracted memories so you can watch the store grow as you chat.

**Workflow inside the UI:** type a question, watch the Tool Trace panel light up tool-by-tool as the agent works, read the final brief in the chat panel, then look at the Memory panel to see what Mem0 just learned about your "client". Try a multi-turn conversation (intake -> legal question -> follow-up) and notice the agent stop re-asking things you already told it.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell. Wait until you see `Running on local URL: http://127.0.0.1:7860` in the output (and the "Mem0: cleared prior memories..." line above it).

1. Open the app: **[http://localhost:7860](http://localhost:7860)**. If the link 404s, the cell hasn't finished starting up yet - give it a few seconds and reload.

1. Try the **same Seattle flooding scenario** you've been running all notebook ("My client in Queen Anne... regraded their lot... February 2024 storm... drainage"). Watch the Tool Trace panel show all five tools fire, then read the brief in the chat panel.

1. Send a **follow-up turn** that does NOT repeat the location or the date ("What evidence should I start collecting?"). Confirm the agent answers without re-asking - that's Mem0 doing its job.

1. When you're done, you can stop the app by clicking the ⏹ stop button in the cell toolbar, or by re-running the cell (which will also reset the memory store).


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Counsel AI - Gradio web UI for the full 5-tool + Mem0 agent
# ─────────────────────────────────────────────────────────────────────
#
# This cell is long but laid out in clear sections - read it top to
# bottom and you'll see the same building blocks from the rest of the
# notebook, just plumbed into a Gradio app:
#
#   1) Create the Counsel agent for the UI (OpenAIChatClient + as_agent
#      with all 5 tools - same pattern as Part 3.7).
#   2) Wire up Mem0 long-term memory under a UI-only user id, wiping
#      that user's memories on every cell run so the app starts fresh.
#   3) Helper functions to retrieve / format memories before each turn
#      and to extract / persist new memories after each turn (same
#      retrieve -> run -> persist lifecycle as Part 3.8).
#   4) A streaming chat handler that runs the agent, captures the
#      print("<tool> was called") trace lines from each tool, and
#      feeds them into the Tool Trace panel as they happen.
#   5) gr.Blocks layout - chat on the left, Tool Trace + Long-term
#      Memory panels on the right.
#   6) demo.launch(server_port=7860, inline=False) - starts the local
#      web server. Open http://localhost:7860 in your browser AFTER
#      this cell prints "Running on local URL: http://127.0.0.1:7860".
# ─────────────────────────────────────────────────────────────────────

# ── 1) Create the Counsel Agent for the UI ──────────────────────────

_ui_client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

# ── Mem0 long-term memory for the UI session ────────────────────────
#
# We re-use the same `memory` client (pgvector in HorizonDB + Azure OpenAI)
# initialised in the Mem0 demo cell above. The UI runs under its own user
# id so it stays isolated from the demo's USER_ID, and we wipe that user's
# memories every time this cell is executed so the launched app always
# starts from a clean slate.
_UI_USER_ID = "ui-counsel-session2"

try:
    memory.delete_all(user_id=_UI_USER_ID)
    print(f"Mem0: cleared prior memories for user '{_UI_USER_ID}'.")
except Exception as _e:
    print(f"Mem0: nothing to clear for user '{_UI_USER_ID}' ({_e}).")


_UI_TOOLS = [
    keyword_case_search,
    semantic_case_search,
    precedent_graph_search,
    case_analyst_extract,
    get_weather_evidence,
]


def _fetch_ui_memory_list(query: str) -> list[str]:
    """Pull the top-k memories Mem0 considers most relevant to the upcoming user turn."""
    hits = memory.search(query=query, user_id=_UI_USER_ID, limit=5).get("results", [])
    return [h["memory"] for h in hits]


def _format_memory_block(items: list[str]) -> str:
    """Render a list of memories into the bullet block injected into the system prompt."""
    if not items:
        return "(no prior memories yet)"
    return "\n".join(f"- {m}" for m in items)


def _build_ui_agent(memory_block: str):
    """Re-build the agent each turn so freshly retrieved memories appear in the system prompt."""
    return _ui_client.as_agent(
        instructions=(
            "You are Counsel, an AI legal associate helping analyze Washington State legal cases.\n\n"
            "## Long-term memory about this client (from prior conversations)\n"
            f"{memory_block}\n\n"
            "Use the memory above to personalize answers - refer to the client by name, skip "
            "intake questions you already know the answer to, and build on facts established "
            "in earlier turns.\n\n"
            
            "If a legal question, this is the required wWorkflow (if a legal question, always run these 4 tools in order):\n"
            "1. keyword_case_search - BM25 full-text search for canonical cases\n"
            "2. semantic_case_search - DiskANN vector search for factually similar cases\n"
            "3. precedent_graph_search - Citation graph traversal via Apache AGE\n"
            "4. case_analyst_extract - Extract structured legal facts via azure_ai.extract\n\n"

            "Weather / Storm / Flooding Tool:\n"
            "5. Always run this tool last if ran. If the question involves weather, flooding, storms, precipitation, drainage, or water damage,\n"
            "use get_weather_evidence to retrieve historical precipitation data for the relevant location and date.\n"
            "Include this data as supporting evidence in your analysis.\n\n"

            "Constraints:\n"
            "- If a legal question, do NOT skip the required steps (1-4).\n"
            "- If any mention of flooding, storms, drainage, or water damage is detected, you must use the Weather / Storm / Flooding Tool. And run this tool last.\n"
            "- Pass outputs between tools (case ids, JSON payloads).\n"
            "- If a legal question, do NOT manually summarize without tools.\n"
            "- If a legal question, call each tool AT MOST ONCE. Never call the same tool twice.\n\n"            

            "(If this is a legal question) Final Response Format (Pretty Markdown):\n"
            "If this is a legal question, after all tool calls complete, synthesize the results into a polished legal analysis.\n"
            "- A short opening paragraph summarizing the key legal holding and its significance.\n"
            "- A short paragraph distinguishing the most important prior cases.\n"
            "- If weather evidence was gathered, a paragraph incorporating precipitation data as supporting evidence.\n"
            "- **Key Takeaway:** A concise statement of the principal legal rule.\n"
            "- **Key Cases in this Line of Authority:** A bulleted list of relevant cases, each with case name, citation, and year.\n"
            "- **Statutes Extracted:** A bulleted list of statutes and what case it is from.\n"
            "- **Holdings Extracted:** A bulleted list of holdings and what case it is from.\n"
        ),
        tools=_UI_TOOLS,
    )

# ── Tool Trace Metadata ──────────────────────────────────────────────
_TOOL_META = {
    "keyword_case_search":    {"idx": 1, "name": "KeywordCaseSearchTool",  "tech": "BM25 Search"},
    "semantic_case_search":   {"idx": 2, "name": "SemanticCaseSearchTool", "tech": "DiskANN with Advanced Filtering"},
    "precedent_graph_search": {"idx": 3, "name": "PrecedentGraphTool",     "tech": "AGE Graph"},
    "case_analyst_extract":   {"idx": 4, "name": "CaseAnalystTool",        "tech": "AI Function azure_ai.extract()"},
    "get_weather_evidence":   {"idx": 5, "name": "WeatherEvidenceTool",    "tech": "Weather External API"},
}

_TRACE_PALETTE = {
    "accent": "#5b5ef7",
    "accent_soft": "#eef0ff",
    "accent_border": "#d8ddff",
    "text": "#1f2340",
    "muted": "#687087",
    "panel": "#ffffff",
    "line": "#dfe3f0",
    "success": "#217346",
    "success_soft": "#edf8f0",
    "success_border": "#cce8d5",
    "success_text": "#1f5f3c",
    "shadow": "0 10px 28px rgba(31, 35, 64, 0.08)",
}

_UI_THEME = gr.themes.Soft(
    primary_hue="indigo",
    neutral_hue="gray",
    radius_size=gr.themes.sizes.radius_lg,
)


def _tool_icon_svg(key: str) -> str:
    accent = _TRACE_PALETTE["accent"]
    icons = {
        "keyword_case_search": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<circle cx="8.5" cy="8.5" r="5.5" stroke="{accent}" stroke-width="1.8"/>'
            f'<path d="M12.5 12.5L16.5 16.5" stroke="{accent}" stroke-width="1.8" stroke-linecap="round"/>'
            f'</svg>'
        ),
        "semantic_case_search": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<ellipse cx="10" cy="5" rx="5.8" ry="2.6" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M4.2 5V10.2C4.2 11.6 6.8 12.8 10 12.8C13.2 12.8 15.8 11.6 15.8 10.2V5" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M4.2 10.2V15C4.2 16.4 6.8 17.6 10 17.6C13.2 17.6 15.8 16.4 15.8 15V10.2" stroke="{accent}" stroke-width="1.6"/>'
            f'</svg>'
        ),
        "precedent_graph_search": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<circle cx="5" cy="10" r="1.7" stroke="{accent}" stroke-width="1.6"/>'
            f'<circle cx="15" cy="5" r="1.7" stroke="{accent}" stroke-width="1.6"/>'
            f'<circle cx="15" cy="15" r="1.7" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M6.5 9L13.2 5.8" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M6.5 11L13.2 14.2" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M15 6.8V13.2" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'</svg>'
        ),
        "case_analyst_extract": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<path d="M6 2.8H11.8L15.5 6.5V16.5C15.5 17.3 14.8 18 14 18H6C5.2 18 4.5 17.3 4.5 16.5V4.3C4.5 3.5 5.2 2.8 6 2.8Z" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M11.5 2.8V6.4H15.2" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M7.5 10H12.5" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M7.5 13H11.2" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'</svg>'
        ),
        "get_weather_evidence": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<path d="M6 14C3.8 14 2 12.4 2 10.4C2 8.7 3.3 7.3 5 7C5.5 4.7 7.5 3 10 3C12.8 3 15 5 15.3 7.5C17 7.8 18 9.2 18 10.8C18 12.6 16.5 14 14.7 14H6Z" stroke="{accent}" stroke-width="1.6" stroke-linejoin="round"/>'
            f'<path d="M8 16V14" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M12 17V14" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M10 18V15" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'</svg>'
        ),
    }
    return icons.get(key, "")


def _header_icon_svg() -> str:
    accent = _TRACE_PALETTE["accent"]
    return (
        f'<svg width="28" height="28" viewBox="0 0 24 24" fill="none" '
        f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
        f'<path d="M4 7H20" stroke="{accent}" stroke-width="1.8" stroke-linecap="round"/>'
        f'<path d="M12 5V19" stroke="{accent}" stroke-width="1.8" stroke-linecap="round"/>'
        f'<path d="M7 7C7 10 5.5 12 4 13.3C5 14.1 6.2 14.5 7.5 14.5C9.4 14.5 10.9 13.4 12 11.8" stroke="{accent}" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>'
        f'<path d="M17 7C17 10 18.5 12 20 13.3C19 14.1 17.8 14.5 16.5 14.5C14.6 14.5 13.1 13.4 12 11.8" stroke="{accent}" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>'
        f'<path d="M9 19H15" stroke="{accent}" stroke-width="1.8" stroke-linecap="round"/>'
        f'</svg>'
    )


# ── Trace Parsing Helpers ────────────────────────────────────────────

def _parse_traces(captured: str) -> list[dict]:
    """Parse captured stdout to identify which tools the agent called."""
    traces, lines = [], captured.split("\n")
    for i, line in enumerate(lines):
        for key, meta in _TOOL_META.items():
            if f"{key} was called" in line:
                details = []
                for nxt in lines[i + 1:]:
                    if nxt.startswith("  "):
                        details.append(nxt.strip())
                    else:
                        break
                traces.append({"key": key, "details": details, **meta})
                break
    return traces


def _new_trace_state() -> dict:
    return {
        "traces": [],
        "line_buffer": "",
        "changed": False,
    }


def _ingest_trace_chunk(state: dict, chunk: str, now: float | None = None) -> None:
    """Update trace state as stdout arrives so the UI stays in sync with actual tool starts."""
    if not chunk:
        return

    if now is None:
        now = time.perf_counter()

    state["line_buffer"] += chunk
    while "\n" in state["line_buffer"]:
        line, state["line_buffer"] = state["line_buffer"].split("\n", 1)
        stripped = line.rstrip("\r")

        matched_key = None
        for key in _TOOL_META:
            if f"{key} was called" in stripped:
                matched_key = key
                break

        if matched_key is not None:
            trace = {"key": matched_key, "details": [], "started_at": now, **_TOOL_META[matched_key]}
            state["traces"].append(trace)
            state["changed"] = True
            continue

        if stripped.startswith("  ") and state["traces"]:
            state["traces"][-1]["details"].append(stripped.strip())
            state["changed"] = True


def _run_ui_agent(message: str, memory_block: str):
    """Build an agent with the latest Mem0 context, then run it on a worker thread."""
    agent = _build_ui_agent(memory_block)
    return asyncio.run(agent.run(message))


def _trace_desc(t: dict) -> str:
    d = " ".join(t.get("details", []))
    k = t["key"]
    if k == "keyword_case_search":
        m = re.search(r"query=(.+?)\s\s+court_level=", d)
        if m:
            q = m.group(1).strip().strip("'\"").replace("\\'", "'").replace('\\"', '"')
        else:
            q = "case law terms"
        q = q[:55]
        return f'Searched for "{q}" and related terms.'
    if k == "semantic_case_search":
        parts = []
        m = re.search(r"court_level=([^\s]+)", d)
        if m and m.group(1) != "None":
            parts.append(f"court_level IN ({m.group(1)})")
        m = re.search(r"min_year=(\d+)", d)
        if m:
            parts.append(f"decision_date >= {m.group(1)}")
        filt = ", ".join(parts) if parts else "no additional filters"
        return f"Expanded search using vector similarity with filters: {filt}."
    if k == "precedent_graph_search":
        m = re.search(r"inputs=(\d+)", d)
        n = m.group(1) if m else "N"
        return f"{n} unique cases found via KeywordCaseSearchTool and SemanticCaseSearchTool."
    if k == "case_analyst_extract":
        fm = re.search(r"fields=([^\s]+)", d)
        cm = re.search(r"case_ids=(\d+)", d)
        n_in = cm.group(1) if cm else "the"
        if fm:
            field_list = [f.strip() for f in fm.group(1).split(",") if f.strip()]
            chips = "".join(
                f'<span style="display:inline-block;padding:2px 8px;margin:2px 4px 2px 0;'
                f'background:#eef0ff;border:1px solid #d8ddff;border-radius:999px;'
                f'font-size:11px;color:#4f56c7;">{f}</span>'
                for f in field_list
            )
            return (
                f'Extracting structured legal entities from {n_in} case opinions:'
                f'<div style="margin-top:6px;">{chips}</div>'
            )
        return f"Extracting structured fields from {n_in} key opinions."
    if k == "get_weather_evidence":
        m = re.search(r"start=(\S+)", d)
        dt = m.group(1) if m else "requested dates"
        return f"Retrieved historical precipitation data for {dt}."
    return ""


def _trace_result(t: dict) -> str:
    d = " ".join(t.get("details", []))
    k = t["key"]
    if k == "keyword_case_search":
        return "Results: 5 cases"
    if k == "semantic_case_search":
        return "Results: 5 cases"
    if k == "precedent_graph_search":
        rm = re.search(r"related_count=(\d+)", d)
        hm = re.search(r"hops=(\d+)", d)
        if rm:
            n = rm.group(1)
            hops = hm.group(1) if hm else "2"
            return f"{n} additional related cases found traversing the graph using {hops} hops"
        return "Paths found via graph traversal"
    if k == "case_analyst_extract":
        cases = re.findall(r"extracted_case=(\d+)\|(.+?)(?=\s+extracted_case=|\s+\w+=|$)", d)
        if cases:
            items = "".join(
                f'<div style="display:flex;align-items:flex-start;gap:6px;margin-top:4px;">'
                f'<span style="color:#217346;font-weight:700;">✓</span>'
                f'<span style="font-size:12px;color:#1f2340;">{name.strip()} '
                f'<span style="color:#687087;">(id {cid})</span></span>'
                f'</div>'
                for cid, name in cases
            )
            return (
                f'<div style="font-weight:600;color:#1f5f3c;margin-top:4px;">'
                f'Extracted {len(cases)} case{"s" if len(cases) != 1 else ""}:</div>'
                f'{items}'
            )
        m = re.search(r"case_ids=(\d+)", d)
        return f"Cases analyzed: {m.group(1) if m else 'N'}"
    return ""


# ── Trace Panel HTML Builder (Light Theme + Flat Icons) ─────────────

def _build_trace_html(traces, t0, running=False, total_time=None):
    """Build trace panel HTML. Supports live updates during streaming."""
    panel = _TRACE_PALETTE["panel"]
    text = _TRACE_PALETTE["text"]
    muted = _TRACE_PALETTE["muted"]
    accent = _TRACE_PALETTE["accent"]
    accent_soft = _TRACE_PALETTE["accent_soft"]
    accent_border = _TRACE_PALETTE["accent_border"]
    line = _TRACE_PALETTE["line"]
    success = _TRACE_PALETTE["success"]
    success_soft = _TRACE_PALETTE["success_soft"]
    success_border = _TRACE_PALETTE["success_border"]
    success_text = _TRACE_PALETTE["success_text"]
    shadow = _TRACE_PALETTE["shadow"]
    now = time.perf_counter()

    if not traces and not running:
        return (
            f'<div style="padding:24px 22px;text-align:center;color:{muted};background:{panel};">'
            f'<div style="width:52px;height:52px;border-radius:16px;background:{accent_soft};margin:0 auto 12px;'
            f'display:flex;align-items:center;justify-content:center;color:{accent};font-size:22px;">⌕</div>'
            f'<div style="font-weight:600;color:{text};margin-bottom:4px;">Tool Trace</div>'
            f'<div>Ask a question to see the research steps</div>'
            f'</div>'
        )
    if not traces and running:
        return (
            f'<div style="padding:24px 22px;text-align:center;color:{muted};background:{panel};">'
            f'<div style="width:52px;height:52px;border-radius:16px;background:{accent_soft};margin:0 auto 12px;'
            f'display:flex;align-items:center;justify-content:center;color:{accent};font-size:22px;" class="spin">◌</div>'
            f'<div style="font-weight:600;color:{text};margin-bottom:4px;">Starting analysis pipeline</div>'
            f'<div>Preparing the first tool call</div>'
            f'</div>'
        )

    n_label = f'{len(traces)} tool{"s" if len(traces) != 1 else ""}'
    state = "running" if running else "used"
    html = (
        f'<div style="background:{panel};padding:2px 2px 8px 2px;">'
        f'<div style="font-weight:700;font-size:15px;color:{text};margin-bottom:14px;">'
        f'Tool Trace <span style="font-weight:500;color:{muted};">({n_label} {state})</span></div>'
    )

    for i, t in enumerate(traces):
        is_last = i == len(traces) - 1
        is_active = running and is_last

        started_at = t.get("started_at", t0)
        if is_active:
            dur = now - started_at
        elif i + 1 < len(traces):
            dur = traces[i + 1].get("started_at", started_at) - started_at
        else:
            end_at = t0 + (total_time or 0)
            dur = end_at - started_at
        dur = max(dur, 0)

        ts = f"{int(dur * 1000)} ms" if dur < 1 else f"{dur:.1f}s"
        desc = _trace_desc(t)
        res = _trace_result(t)
        icon = _tool_icon_svg(t["key"])

        if is_active:
            status_icon = f'<span style="color:{accent};font-size:14px;" class="spin">◌</span>'
            row_bg = "#f8f9ff"
        else:
            status_icon = f'<span style="color:{success};font-size:14px;">●</span>'
            row_bg = panel

        connector = ""
        if i < len(traces) - 1:
            connector = (
                f'<div style="position:absolute;left:15px;top:34px;width:2px;height:40px;'
                f'background:{accent_border};border-radius:999px;"></div>'
            )

        res_div = (
            f'<div style="color:{muted};font-size:13px;margin-top:6px;">{res}</div>'
            if res and (not is_active or t["key"] == "case_analyst_extract") else ""
        )

        html += (
            f'<div style="display:flex;gap:12px;position:relative;margin-bottom:14px;">'
            f'<div style="position:relative;width:32px;flex:0 0 32px;display:flex;justify-content:center;">'
            f'<div style="width:30px;height:30px;border-radius:999px;background:{accent};color:#fff;' 
            f'display:flex;align-items:center;justify-content:center;font-weight:700;font-size:13px;">{t["idx"]}</div>'
            f'{connector}</div>'
            f'<div style="flex:1;background:{row_bg};border:1px solid {line};border-radius:14px;padding:12px 14px;'
            f'box-shadow:{shadow};">'
            f'<div style="display:flex;justify-content:space-between;align-items:flex-start;gap:12px;">'
            f'<div style="display:flex;gap:10px;align-items:flex-start;min-width:0;">'
            f'<div style="width:20px;height:20px;display:flex;align-items:center;justify-content:center;flex:0 0 20px;">{icon}</div>'
            f'<div style="min-width:0;">'
            f'<div style="font-weight:700;color:{text};line-height:1.25;">{t["name"]} '
            f'<span style="font-weight:500;color:{muted};font-size:12px;">({t["tech"]})</span></div>'
            f'<div style="color:{muted};font-size:13px;line-height:1.45;margin-top:4px;">{desc}</div>'
            f'{res_div}</div></div>'
            f'<div style="text-align:right;white-space:nowrap;color:{muted};font-size:13px;">{ts}<div style="margin-top:4px;">{status_icon}</div></div>'
            f'</div></div></div>'
        )

    if running:
        elapsed = now - t0
        html += (
            f'<div style="display:flex;align-items:center;gap:8px;padding:12px 14px;border-radius:12px;' 
            f'background:{accent_soft};border:1px solid {accent_border};">'
            f'<span class="spin" style="color:{text};">◌</span>'
            f'<span style="font-weight:700;color:{text};">Running... {elapsed:.1f}s elapsed</span></div>'
        )
    else:
        t_disp = total_time or 0
        html += (
            f'<div style="display:flex;align-items:center;gap:8px;padding:12px 14px;border-radius:12px;' 
            f'background:{success_soft};border:1px solid {success_border};color:{success_text};">'
            f'<span style="font-size:14px;color:{success};"></span>'
            f'<span style="font-weight:700;color:{success_text};">Completed in {t_disp:.2f} seconds</span></div>'
        )

    html += "</div>"
    return html


# ── Memory Panel HTML Builder ────────────────────────────────────────

def _build_memory_html(retrieved: list[str], new_facts: list[str], capturing: bool = False) -> str:
    """Render the Mem0 side panel showing memories retrieved + captured this turn."""
    text = _TRACE_PALETTE["text"]
    muted = _TRACE_PALETTE["muted"]
    accent = _TRACE_PALETTE["accent"]
    accent_soft = _TRACE_PALETTE["accent_soft"]
    accent_border = _TRACE_PALETTE["accent_border"]
    line = _TRACE_PALETTE["line"]
    success = _TRACE_PALETTE["success"]
    success_soft = _TRACE_PALETTE["success_soft"]
    success_border = _TRACE_PALETTE["success_border"]
    success_text = _TRACE_PALETTE["success_text"]

    # Empty state - shown on first render before any turn has happened.
    if not retrieved and not new_facts and not capturing:
        return (
            f'<div style="padding:6px 0;text-align:center;color:{muted};">'
            f'<div style="width:42px;height:42px;border-radius:14px;background:{accent_soft};margin:0 auto 10px;'
            f'display:flex;align-items:center;justify-content:center;color:{accent};font-size:20px;">◇</div>'
            f'<div style="font-weight:600;color:{text};margin-bottom:4px;">Mem0 Long-Term Memory</div>'
            f'<div style="font-size:13px;">Memories retrieved and captured will appear here each turn.</div>'
            f'</div>'
        )

    html = (
        f'<div style="font-weight:700;font-size:15px;color:{text};margin-bottom:12px;">'
        f'Mem0 Long-Term Memory '
        f'<span style="font-weight:500;color:{muted};font-size:12px;">(user: {_UI_USER_ID})</span>'
        f'</div>'
    )

    # Section 1: memories retrieved for this turn
    html += (
        f'<div style="font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:0.04em;'
        f'color:{accent};margin:4px 0 8px;">Retrieved for this turn ({len(retrieved)})</div>'
    )
    if retrieved:
        for m in retrieved:
            html += (
                f'<div style="padding:8px 10px;background:{accent_soft};border:1px solid {accent_border};'
                f'border-radius:10px;margin-bottom:6px;font-size:13px;color:{text};line-height:1.4;">'
                f'<span style="color:{accent};font-weight:700;margin-right:6px;">↳</span>{m}</div>'
            )
    else:
        html += (
            f'<div style="padding:8px 10px;background:#f7f8fc;border:1px dashed {line};border-radius:10px;'
            f'font-size:13px;color:{muted};">(no prior memories yet)</div>'
        )

    # Section 2: memories captured from this turn
    if capturing:
        html += (
            f'<div style="font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:0.04em;'
            f'color:{muted};margin:14px 0 8px;display:flex;align-items:center;gap:6px;">'
            f'<span class="spin" style="color:{muted};">◌</span> Capturing new memories...</div>'
            f'<div style="padding:8px 10px;background:#f7f8fc;border:1px dashed {line};border-radius:10px;'
            f'font-size:13px;color:{muted};">Waiting for the agent to finish before Mem0 extracts facts.</div>'
        )
    else:
        html += (
            f'<div style="font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:0.04em;'
            f'color:{success};margin:14px 0 8px;">Captured this turn ({len(new_facts)})</div>'
        )
        if new_facts:
            for m in new_facts:
                html += (
                    f'<div style="padding:8px 10px;background:{success_soft};border:1px solid {success_border};'
                    f'border-radius:10px;margin-bottom:6px;font-size:13px;color:{success_text};line-height:1.4;">'
                    f'<span style="color:{success};font-weight:700;margin-right:6px;">+</span>{m}</div>'
                )
        else:
            html += (
                f'<div style="padding:8px 10px;background:#f7f8fc;border:1px dashed {line};border-radius:10px;'
                f'font-size:13px;color:{muted};">(no new facts extracted from this turn)</div>'
            )

    return html


# ── Chat Handler (async generator for live streaming) ────────────────

async def _chat_fn(message: str, history: list):
    if not message or not message.strip():
        yield history, _build_trace_html([], 0), _build_memory_html([], []), ""
        return

    # Mem0: retrieve relevant memories BEFORE we start tee'ing stdout so the
    # tool-trace parser doesn't see these lines. The memory block is injected
    # into the agent's system prompt for this turn.
    retrieved = _fetch_ui_memory_list(message)
    memory_block = _format_memory_block(retrieved)
    print("\n--- Mem0: memories retrieved for this turn ---")
    print(memory_block)

    old_stdout = sys.stdout
    buf = io.StringIO()
    trace_state = _new_trace_state()

    class _Tee:
        def write(self, s):
            buf.write(s)
            _ingest_trace_chunk(trace_state, s)
            old_stdout.write(s)
        def flush(self):
            buf.flush()
            old_stdout.flush()
        def __getattr__(self, name):
            return getattr(old_stdout, name)
    sys.stdout = _Tee()

    t0 = time.perf_counter()
    pending_history = history + [{"role": "user", "content": message}]
    # Show retrieved memories in the side panel immediately; new-memory
    # section is left in "capturing..." state until the agent finishes.
    memory_html = _build_memory_html(retrieved, [], capturing=True)
    yield pending_history, _build_trace_html([], t0, running=True), memory_html, ""

    task = asyncio.create_task(asyncio.to_thread(_run_ui_agent, message, memory_block))
    last_render = 0.0

    while not task.done():
        await asyncio.sleep(0.15)
        now = time.perf_counter()
        should_render = trace_state["changed"] or trace_state["traces"] or (now - last_render >= 0.5)
        if not should_render:
            continue

        trace_state["changed"] = False
        last_render = now
        trace_html = _build_trace_html(trace_state["traces"], t0, running=True)
        yield pending_history, trace_html, memory_html, ""

    elapsed = time.perf_counter() - t0
    sys.stdout = old_stdout

    try:
        res = task.result()
        response = res.text
    except Exception as e:
        response = f"⚠️ An error occurred: {e}"

    # Mem0: persist this turn so the next message can recall it. Done after
    # stdout is restored so the prints go straight to the notebook output
    # (and not into the tool-trace buffer).
    new_facts: list[str] = []
    if not response.startswith("⚠️"):
        try:
            added = memory.add(
                [
                    {"role": "user",      "content": message},
                    {"role": "assistant", "content": response},
                ],
                user_id=_UI_USER_ID,
            )
            new_facts = [m["memory"] for m in added.get("results", [])]
            print("--- Mem0: new memories extracted from this turn ---")
            for f in new_facts or ["(none)"]:
                print(f"+ {f}")
        except Exception as _e:
            print(f"Mem0: failed to persist this turn ({_e}).")

    parsed = _parse_traces(buf.getvalue())
    if len(parsed) > len(trace_state["traces"]):
        for i, parsed_trace in enumerate(parsed):
            if i < len(trace_state["traces"]):
                continue
            parsed_trace["started_at"] = t0 + elapsed
            trace_state["traces"].append(parsed_trace)

    trace_html = _build_trace_html(trace_state["traces"], t0, running=False, total_time=elapsed)
    memory_html = _build_memory_html(retrieved, new_facts, capturing=False)

    final_history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response},
    ]
    yield final_history, trace_html, memory_html, ""


# ── Status Bar ───────────────────────────────────────────────────────

def _status_html() -> str:
    now = datetime.now().strftime("%I:%M %p")
    text = _TRACE_PALETTE["muted"]
    success = _TRACE_PALETTE["success"]
    return (
        f'<div style="display:flex;align-items:center;gap:6px;padding:8px 16px;'
        f'font-size:12px;color:{text};flex-wrap:wrap;">'
        f'Connected to HorizonDB (Reader Endpoint) '
        f'<span style="color:{success};">●</span>'
        f'<span style="margin:0 6px;">|</span> Database: {DB_CONFIG.get("dbname", "")}'
        f'<span style="margin:0 6px;">|</span> User: {DB_CONFIG.get("user", "")}'
        f'<span style="margin:0 6px;">|</span> Time: {now}</div>'
    )


# ── Sample Prompts ───────────────────────────────────────────────────
# These mirror the four TURN prompts from the Mem0 demo cell above, so
# students can click through the same conversation flow inside the UI and
# watch the memory panel light up turn by turn. The 5th prompt is a
# deliberately silly one to show how the agent handles novel fact patterns.

_PROMPT_1 = (
    "Hi, I am an attorney and my client owns a property in the 'Queen Anne neighborhood' of Seattle, Washington. "
    "Please remember those details for later. No legal question yet."
)
_PROMPT_2 = (
    "After a heavy storm during April and May of 2024, my client's yard now floods every time it rains. "
    "A neighbor of my client regraded their lot last February of 2024, and the city repaved the road at the same time period. "
    "Does my client have a claim?"
)
_PROMPT_3 = (
    "Given what you found, who is the most promising defendant and what "
    "evidence should I start collecting?"
)
_PROMPT_4 = (
    "I have a second client, whose condo in downtown Seattle, Washington had a spaceship crash into it, "
    "and also destroy the traffic lights on the street. What claims does my client have?"
)


# ── Build the Gradio App ────────────────────────────────────────────

_CSS = """
:root {
    --counsel-bg: #f6f7fb;
    --counsel-panel: #ffffff;
    --counsel-line: #e2e7f3;
    --counsel-text: #1f2340;
    --counsel-muted: #687087;
    --counsel-accent: #5b5ef7;
    --counsel-accent-soft: #eef0ff;
    --counsel-font: Aptos, "Segoe UI", Tahoma, Geneva, Verdana, sans-serif;
}

.gradio-container,
.gradio-container * {
    font-family: var(--counsel-font) !important;
}

.gradio-container {
    background: linear-gradient(180deg, #fbfbfe 0%, #f4f6fb 100%);
}

.counsel-shell {
    background: transparent;
}

.counsel-hdr {
    padding: 2px 2px 10px 2px;
}

.counsel-hdr h1 {
    margin: 0;
    font-size: 24px;
    display: flex;
    align-items: center;
    gap: 10px;
    color: var(--counsel-text);
}

.counsel-hdr p {
    margin: 4px 0 0;
    font-size: 14px;
    color: var(--counsel-muted);
}

.counsel-header-icon {
    width: 30px;
    height: 30px;
    display: inline-flex;
    align-items: center;
    justify-content: center;
    flex: 0 0 30px;
}

.counsel-chat,
.counsel-trace,
.counsel-memory {
    border: 1px solid var(--counsel-line);
    border-radius: 18px;
    background: var(--counsel-panel);
    box-shadow: 0 16px 34px rgba(31, 35, 64, 0.06);
}

.counsel-chat {
    overflow: hidden;
}

.counsel-trace,
.counsel-memory {
    padding: 18px;
}

.counsel-memory {
    margin-top: 12px;
}

.counsel-right-stack {
    background: transparent;
    border: 0;
    box-shadow: none;
    padding: 0;
}

.counsel-chatbot,
.counsel-chatbot > .wrap,
.counsel-chatbot .wrap,
.counsel-chatbot .bubble-wrap,
.counsel-chatbot .message-wrap,
.counsel-chatbot .message-row,
.counsel-chatbot .panel,
.counsel-chatbot .scroll-hide {
    background: #ffffff !important;
}

.counsel-chatbot {
    border: 0 !important;
}

.counsel-chatbot .message,
.counsel-chatbot .message.user,
.counsel-chatbot .message.bot,
.counsel-chatbot .bubble {
    background: transparent !important;
    color: var(--counsel-text) !important;
    border: 0 !important;
    box-shadow: none !important;
}

.counsel-chatbot .message-content {
    background: #ffffff !important;
    color: var(--counsel-text) !important;
    border: 1px solid var(--counsel-line) !important;
    box-shadow: none !important;
}

.counsel-chatbot .message.user {
    background: #f4f6ff !important;
}

.counsel-chatbot .message.bot {
    background: #ffffff !important;
}

.counsel-chatbot .message * {
    color: var(--counsel-text) !important;
}

.counsel-chatbot .avatar-container,
.counsel-chatbot .avatar-container * {
    background: #f4f6ff !important;
    color: var(--counsel-accent) !important;
}

.counsel-examples {
    align-items: center;
    gap: 8px;
    flex-wrap: wrap;
}

.counsel-examples .gr-button {
    border: 1px solid #dde2ff;
    background: #f7f8ff;
    color: #4f56c7;
    border-radius: 10px;
}

.counsel-input-row {
    gap: 12px;
    align-items: center;
}

.counsel-input,
.counsel-input > div,
.counsel-input .wrap,
.counsel-input .block {
    background: #ffffff !important;
    border-radius: 14px !important;
}

.counsel-input textarea,
.counsel-input input,
.counsel-input-row textarea,
.counsel-input-row input {
    background: #ffffff !important;
    border: 1px solid var(--counsel-line) !important;
    border-radius: 14px !important;
    color: var(--counsel-text) !important;
    box-shadow: none !important;
}

.counsel-input textarea::placeholder,
.counsel-input input::placeholder {
    color: #8f96ab !important;
}

.counsel-send {
    border-radius: 14px !important;
}

footer { display: none !important; }
@keyframes spin { from { transform: rotate(0deg); } to { transform: rotate(360deg); } }
.spin { display: inline-block; animation: spin 1s linear infinite; }
"""

with gr.Blocks(title="Counsel AI – Legal Research Assistant", css=_CSS, theme=_UI_THEME) as demo:
    with gr.Column(elem_classes=["counsel-shell"]):
        gr.HTML(
            '<div class="counsel-hdr">'
            f'<h1><span class="counsel-header-icon">{_header_icon_svg()}</span> Counsel AI - Legal Research Assistant</h1>'
            '<p>Ask legal questions. Counsel uses HorizonDB to find, analyze, and synthesize case law.</p>'
            '</div>'
        )

        with gr.Row(equal_height=True):
            with gr.Column(scale=3, elem_classes=["counsel-chat"]):
                chatbot = gr.Chatbot(height=500, show_label=False, type="messages", elem_classes=["counsel-chatbot"])
            with gr.Column(scale=2, elem_classes=["counsel-right-stack"]):
                with gr.Column(elem_classes=["counsel-trace"]):
                    trace_panel = gr.HTML(value=_build_trace_html([], 0))
                with gr.Column(elem_classes=["counsel-memory"]):
                    memory_panel = gr.HTML(value=_build_memory_html([], []))

        with gr.Row(elem_classes=["counsel-examples"]):
            gr.HTML('<span style="font-size:13px;color:#687087;padding:0 4px;">Example questions</span>')
            ex1 = gr.Button("1. Queen Anne client", size="sm")
            ex2 = gr.Button("2. Yard flooding after neighbor regrade & city repave", size="sm")
            ex3 = gr.Button("3. Most promising defendant & evidence to collect", size="sm")            
            ex4 = gr.Button("4. Spaceship crashed into a second client's condo", size="sm")

        with gr.Row(elem_classes=["counsel-input-row"]):
            msg = gr.Textbox(
                placeholder="Ask a legal question...",
                show_label=False,
                scale=5,
                container=False,
                elem_classes=["counsel-input"],
            )
            send_btn = gr.Button("Send", variant="primary", scale=1, elem_classes=["counsel-send"])

        gr.HTML(_status_html())

    send_btn.click(_chat_fn, [msg, chatbot], [chatbot, trace_panel, memory_panel, msg])
    msg.submit(_chat_fn, [msg, chatbot], [chatbot, trace_panel, memory_panel, msg])

    ex1.click(lambda: _PROMPT_1, outputs=msg)
    ex2.click(lambda: _PROMPT_2, outputs=msg)
    ex3.click(lambda: _PROMPT_3, outputs=msg)
    ex4.click(lambda: _PROMPT_4, outputs=msg)    

gr.close_all()
demo.launch(server_port=7860, inline=False)


## 🎉 Congratulations you have completed the Lab!

#### 🚀 Next Steps

We have curated additional resources to enhance your ongoing journey in building AI agents and AI-powered applications with Azure Database for PostgreSQL.

- A more detailed blog post about the legal case example of lab in the [GraphRAG Solution for Azure Database for PostgreSQL](https://aka.ms/pg-graphrag) check the code in the [GitHub repository](https://aka.ms/postgres-graphrag-solution).
- Learn more about [Graph data in Azure Database for PostgreSQL](https://aka.ms/age-blog).
- Get familiar with the new [PostgreSQL extension for Visual Studio Code]().